# Portrait-to-Anime Translation: Compact CycleGAN Baseline

**Author:** Ashwin Parthasarathy  

This notebook trains an unpaired photo-to-anime portrait translator using PyTorch. A compact CycleGAN learns both translation directions, although the intended application is photo-to-anime. The baseline establishes whether the pipeline learns recognizable stylization and exposes failure cases before a stronger architecture is evaluated.


## Dataset and Approach

This project uses Selfie2Anime, an unpaired dataset containing real face photographs and anime portraits. Each domain includes 3,400 training images and 100 test images at 256 × 256 resolution. The images are available through the [official U-GAT-IT repository](https://github.com/taki0112/UGATIT). Photo and anime images do not depict corresponding subjects, so the model must learn translation without paired examples.

The supplied test partitions are reserved for final evaluation. From each training domain, 340 images are set aside for validation, leaving 3,060 images per domain for training. Automated checks verify image readability, dimensions, and exact duplicates; they do not establish separation by identity or near-duplicate content.

A compact CycleGAN serves as the baseline. Two generators learn photo-to-anime and anime-to-photo translation, while two discriminators distinguish generated images from real images in their respective domains. Adversarial loss encourages target-domain appearance, cycle-consistency loss encourages reconstruction after translation in both directions, and identity loss discourages unnecessary changes to images already in the target domain.

The baseline trains at 128 × 128 resolution to measure feasibility on local hardware. Training runs for 20 epochs after 20 diagnostic optimizer updates, with checkpoints saved at epochs 10 and 20. Fixed validation portraits support comparisons of stylization, facial structure, expressions, accessories, and unwanted color changes. U-GAT-IT at 256 × 256 is planned as a subsequent experiment.

## 1. Environment, reproducibility, and run configuration

All experiment settings are declared here. CUDA, Apple MPS, and CPU are supported. Separate run directories prevent a fresh execution from overwriting the baseline checkpoints.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from zipfile import ZipFile
import hashlib
from itertools import chain
from time import perf_counter
import shutil
import time
from datetime import datetime, timezone
from IPython.display import display

In [ ]:
PROJECT_ROOT = Path.cwd()
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use an available GPU backend for training; otherwise fall back to CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else "cpu"
)

for folder in [
    "data/raw", # Original dataset archive and extracted images.
    "data/processed", # Image inventory and reproducible split manifests.
    "figures", # 
    "reports",
    "src",
    "checkpoints"
]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Device:", device)

# Set the configuration before running the notebook.
RUN_MODE = "train"  # "train" or "evaluate"

IMAGE_SIZE = 128 # Downsample the 256-pixel originals for the baseline.
BATCH_SIZE = 1 # Process one unpaired photo/anime sample per update.
PILOT_EPOCHS = 20
SMOKE_STEPS = 20 # Actual training updates before epoch 1; also measure speed.
BASE_CHANNELS = 32 # Initial convolution width; controls model capacity.
RESIDUAL_BLOCKS = 4 # Residual transformations in each generator's bottleneck.
LEARNING_RATE = 0.0002 # Shared initial learning rate for both optimizers.

# Weight reconstruction terms relative to the adversarial objective.
LAMBDA_CYCLE = 10.0 # Penalize differences after translating to the other domain and back.
LAMBDA_IDENTITY = 5.0 # Penalize changes to images already in the target domain

assert RUN_MODE in {"train", "evaluate"}

if RUN_MODE == "train":
    run_id = datetime.now(timezone.utc).strftime("baseline_%Y%m%dT%H%M%S_%fZ")
    RUN_DIR = PROJECT_ROOT / "runs" / run_id
else:
    RUN_DIR = Path(EVALUATION_RUN_DIR)
    if not RUN_DIR.is_absolute():
        RUN_DIR = PROJECT_ROOT / RUN_DIR
    assert RUN_DIR.is_dir(), f"Missing evaluation directory: {RUN_DIR}"

for folder in ["figures", "reports", "checkpoints"]:
    (RUN_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Mode:", RUN_MODE)
print("Run directory:", RUN_DIR)

## 2. Dataset acquisition and archive inspection

Selfie2Anime provides unpaired domains: A contains real portraits and B contains anime portraits. Matching positions or filenames do not indicate corresponding identities. This cell inspects the archive without modifying it.

In [ ]:
DATASET_ZIP = PROJECT_ROOT / "data/raw/selfie2anime.zip"
assert DATASET_ZIP.is_file(), f"File not found: {DATASET_ZIP}"

with ZipFile(DATASET_ZIP) as archive:
    image_members = [
        name for name in archive.namelist()
        if Path(name).suffix.lower() in {".jpg", ".jpeg", ".png"}
        and not name.startswith("__MACOSX/")
    ]

print("Images in archive:", len(image_members))
print("\nFirst five paths:")
print("\n".join(image_members[:5]))

## 2.1. Extract and count the supplied partitions

Extraction validates member destinations first. The supplied test partitions are retained for eventual final evaluation, not used to select this baseline.

In [ ]:
DATASET_DIR = PROJECT_ROOT / "data/raw/selfie2anime"

with ZipFile(DATASET_ZIP) as archive:
    # Validate paths before extracting.
    destination = DATASET_DIR.resolve()
    for member in archive.infolist():
        target = (destination / member.filename).resolve()
        if not target.is_relative_to(destination):
            raise ValueError(f"Unsafe archive path: {member.filename}")

    archive.extractall(DATASET_DIR)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

for partition in ["trainA", "trainB", "testA", "testB"]:
    folder = DATASET_DIR / partition
    paths = [
        path for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ]
    print(f"{partition}: {len(paths)}")

## 2.2. Representative training images

A reproducible sample of eight images per domain checks framing and domain mapping. The data includes varied image quality and a limited demographic/style scope; these samples are not a demographic audit.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 5))

for row_index, partition in enumerate(["trainA", "trainB"]):
    paths = sorted(
        path for path in (DATASET_DIR / partition).rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    samples = random.Random(SEED).sample(paths, 8)

    for axis, path in zip(axes[row_index], samples):
        with Image.open(path) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(partition, fontsize=10)
        axis.axis("off")

plt.tight_layout()
plt.show()

## 3. Automated Image Integrity and Exact-Duplicate Audit

Before creating validation splits, every image is opened and fully decoded to confirm that it is readable. Its dimensions are recorded, and an RGB representation is used for a consistent comparison across image formats.

A SHA-256 hash is calculated from each image’s dimensions and decoded RGB pixels. Comparing these hashes within each domain identifies exact duplicates, even when filenames or file metadata differ. Re-encoded images are detected only if their decoded pixels remain identical.

Duplicate checks serve two purposes: preventing identical images from appearing across training, validation, and test splits, and avoiding repeated training examples that unintentionally give certain images more influence.

This audit detects exact pixel matches. It does not establish whether different images show the same person or anime character, nor does it detect near duplicates such as resized, cropped, or slightly edited copies. The original image files remain unchanged; the inventory and audit results are recorded separately.

In [ ]:
inventory_rows = []
image_errors = []

for partition in ["trainA", "trainB", "testA", "testB"]:
    paths = sorted(
        path for path in (DATASET_DIR / partition).rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

    for path in paths:
        try:
            with Image.open(path) as image:
                rgb = image.convert("RGB")
                rgb.load()
                pixel_hash = hashlib.sha256(f"{rgb.size}".encode() + rgb.tobytes()).hexdigest()

                inventory_rows.append({
                    "path": str(path.relative_to(PROJECT_ROOT)),
                    "partition": partition,
                    "domain": "photo" if partition.endswith("A") else "anime",
                    "width": rgb.width,
                    "height": rgb.height,
                    "pixel_sha256": pixel_hash
                })
        except Exception as error:
            image_errors.append((str(path), str(error)))

image_inventory = pd.DataFrame(inventory_rows)
image_inventory.to_csv(PROJECT_ROOT / "data/processed/image_inventory.csv", index=False)

print("Unreadable images:", len(image_errors))
print("\nImage dimensions:")
print(image_inventory.groupby(["width", "height"]).size())

duplicate_rows = image_inventory.loc[
    image_inventory.duplicated(["domain", "pixel_sha256"], keep=False)
]
print("\nImages belonging to exact-duplicate sets:", len(duplicate_rows))

if not duplicate_rows.empty:
    display(duplicate_rows[["partition", "path", "pixel_sha256"]])

In [ ]:
# Stop rather than silently omit corrupt images from the experiment.
assert not image_errors, f"Resolve unreadable images before training: {image_errors[:3]}"
assert len(image_inventory) == 7000, "Unexpected dataset inventory; inspect before proceeding."


## 4. Training, Validation, and Test Manifests

The dataset provides separate training and test partitions for real photographs and anime portraits. The supplied test partitions are preserved, while 340 images are randomly selected from each training domain to create validation partitions. A fixed random seed makes this selection reproducible when the same image inventory is used.

The resulting allocation is:

| Partition | Real photographs | Anime portraits | Purpose |
|---|---:|---:|---|
| Training | 3,060 | 3,060 | Update generator and discriminator weights |
| Validation | 340 | 340 | Monitor generated outputs and compare model choices |
| Test | 100 | 100 | Evaluate the selected model after development |

The domains remain unpaired: a photograph and an anime portrait in the same partition do not represent matching subjects. Each domain is partitioned independently, and no photo-to-anime correspondence is assumed.

Validation images do not contribute to gradient updates. They support decisions about training duration, checkpoint selection, and subsequent experiments. The test images remain unused during these decisions so that final evaluation is separate from model development.

Before saving the manifest, checks confirm that each image path appears only once and that no exact pixel hash occurs in multiple splits within a domain. These checks prevent exact-image overlap, but they do not establish identity independence: different photographs of the same person, repeated anime characters, or near-duplicate images may still appear across partitions.

The manifest records each image’s relative path, source partition, domain, dimensions, pixel hash, and assigned split. Relative paths allow the same manifest to be used from another checkout of the repository without depending on this machine’s absolute directory structure. The image files themselves are not moved or modified.

In [ ]:
split_manifest = image_inventory.copy()
split_manifest["split"] = "test"

for domain in ["photo", "anime"]:
    training_indices = split_manifest.index[
        split_manifest["domain"].eq(domain)
        & split_manifest["partition"].str.startswith("train")
    ]

    validation_indices = (
        split_manifest.loc[training_indices]
        .sample(n=340, random_state=SEED)
        .index
    )

    split_manifest.loc[training_indices, "split"] = "train"
    split_manifest.loc[validation_indices, "split"] = "validation"

assert split_manifest["path"].is_unique
assert (
    split_manifest.groupby(["domain", "pixel_sha256"])["split"]
    .nunique()
    .max()
    == 1
)

split_manifest.to_csv(
    PROJECT_ROOT / "data/processed/split_manifest.csv",
    index=False
)

display(pd.crosstab(split_manifest["domain"], split_manifest["split"]))

## 5. Image Preprocessing and Unpaired Sampling

Images are converted to RGB and resized from 256 × 256 to 128 × 128 pixels for the baseline experiment. The smaller resolution reduces computation and memory requirements, allowing training behavior to be assessed quickly on local hardware. This also removes fine detail, which may limit the quality of facial features and textures in generated portraits.

### Training and Evaluation Transforms

Training images receive a random horizontal flip with a probability of 0.5. The transformation is applied independently to each photograph and anime portrait, introducing variation in orientation without requiring additional images.

Validation images are resized and normalized using the same settings, but receive no random augmentation. This keeps evaluation inputs consistent when comparing checkpoints.

After conversion to tensors, RGB intensities lie between 0 and 1. Normalization with a mean and standard deviation of 0.5 for each channel maps them to the range [-1, 1]:

$$
x_{\text{normalized}} = \frac{x - 0.5}{0.5} = 2x - 1
$$

This range matches the generator’s final `tanh` activation. Before displaying generated images, the normalization is reversed to recover values in [0, 1].

### Independent Sampling Across Domains

The dataset contains no corresponding photo–anime pairs. During training, each photograph is combined with an independently sampled anime portrait. These two images provide examples of their respective domains; the anime image is not a desired reconstruction of that particular photograph.

With equal training-domain sizes and a batch size of one, each epoch visits all 3,060 photographs once in shuffled order. Anime portraits are sampled with replacement, so an individual anime image may be selected multiple times or not selected at all during an epoch. This avoids imposing a fixed correspondence between unrelated images.

### Data Loader Configuration

A batch size of one processes one photograph and one independently selected anime portrait per training step, limiting memory use. Each domain’s batch has shape `(1, 3, 128, 128)`, representing the batch size, RGB channels, height, and width.

The data loader uses `num_workers=0`, so image loading occurs in the notebook’s main process. This simplifies execution across notebook environments and avoids worker-process issues, although it does not provide parallel data loading. A seeded generator controls photograph shuffling, while the seeded Python and PyTorch random-number generators control anime sampling and augmentation.

In [ ]:
class UnpairedPortraitDataset(Dataset):
    """Load independently sampled photo and anime training images.

    Parameters
    ----------
    manifest : pandas.DataFrame
        Training rows containing 'path' and 'domain' columns.
    root : pathlib.Path
        Project directory used to resolve image paths.
    transform : callable
        Image preprocessing applied independently to each domain.
    """

    def __init__(self, manifest, root, transform):
        self.photo_paths = sorted(
            manifest.loc[manifest["domain"].eq("photo"), "path"].tolist()
        )
        self.anime_paths = sorted(
            manifest.loc[manifest["domain"].eq("anime"), "path"].tolist()
        )
        if not self.photo_paths or not self.anime_paths:
            raise ValueError("Both image domains must contain training images.")

        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        """Return the number of samples per training epoch."""
        return max(len(self.photo_paths), len(self.anime_paths))

    def __getitem__(self, index):
        """Return a dictionary of photo and anime tensors for one sample.

        Parameters
        ----------
        index : int
            Index used to select the photo.

        Returns
        -------
        dict[str, torch.Tensor]
            Independently transformed images under 'photo' and 'anime'.
        """
        photo_path = self.photo_paths[index % len(self.photo_paths)]
        # Independent sampling prevents an accidental fixed photo/anime pairing.
        anime_path = random.choice(self.anime_paths)

        with Image.open(self.root / photo_path) as image:
            photo = self.transform(image.convert("RGB"))

        with Image.open(self.root / anime_path) as image:
            anime = self.transform(image.convert("RGB"))

        return {"photo": photo, "anime": anime}

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3)
])

train_dataset = UnpairedPortraitDataset(
    split_manifest.loc[split_manifest["split"].eq("train")],
    PROJECT_ROOT,
    train_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED)
)

In [ ]:
batch = next(iter(train_loader))

for domain, tensor in batch.items():
    print(
        f"{domain}: shape={tuple(tensor.shape)}, "
        f"range=[{tensor.min().item():.3f}, {tensor.max().item():.3f}]"
    )

## 6. Model Architecture and Initialization

The model uses a compact CycleGAN architecture to learn translation between real photographs and anime portraits without paired examples. It contains four networks:

| Network | Role |
|---|---|
| Photo-to-anime generator | Translates real portraits into anime-style portraits |
| Anime-to-photo generator | Translates anime portraits into photographic-style portraits |
| Photo discriminator | Distinguishes real photographs from generated photographs |
| Anime discriminator | Distinguishes real anime portraits from generated anime portraits |

Although photo-to-anime translation is the intended application, both generators are trained. The reverse generator enables cycle-consistency training: an image is translated into the other domain and then translated back to reconstruct its original content ([Zhu et al., 2017](https://arxiv.org/abs/1703.10593)).

### Generator Architecture

Both generators share the same architecture but learn separate weights. Each generator contains three stages:

1. **Encoding:** A 7 × 7 convolution maps the RGB input to 32 feature channels. Two stride-2 convolutions then reduce spatial resolution while increasing the channel count, allowing subsequent layers to operate on a smaller feature representation.
2. **Residual transformation:** Four residual blocks transform the encoded features. Each block adds its learned transformation to its input, providing a direct path for information and gradients through the block.
3. **Decoding:** Two stages of nearest-neighbor upsampling followed by convolution restore the original spatial resolution. A final 7 × 7 convolution produces three output channels, and a `tanh` activation bounds their values to [-1, 1].

For the 128 × 128 baseline, the main feature shapes are:

| Stage | Channels | Spatial dimensions |
|---|---:|---|
| RGB input | 3 | 128 × 128 |
| Initial convolution | 32 | 128 × 128 |
| First downsampling stage | 64 | 64 × 64 |
| Second downsampling stage | 128 | 32 × 32 |
| Four residual blocks | 128 | 32 × 32 |
| First upsampling stage | 64 | 64 × 64 |
| Second upsampling stage | 32 | 128 × 128 |
| RGB output | 3 | 128 × 128 |

Instance normalization and ReLU activations are used throughout the intermediate generator layers. Reflection padding supplies boundary values for the initial, residual, and decoding convolutions. Upsampling and convolution are implemented as separate operations.

### Discriminator Architecture

Each discriminator uses a [PatchGAN design (Isola et al., 2017)](https://arxiv.org/abs/1611.07004), producing scores for overlapping image regions rather than a single score for the entire image. For a 128 × 128 input, its output has shape `(N, 1, 14, 14)`, where `N` is the batch size.

The discriminator begins with 32 feature channels and increases the channel count through successive convolutions. Intermediate layers use instance normalization and LeakyReLU activations. The final convolution produces unbounded scores without a sigmoid activation because training uses a least-squares adversarial objective. These scores are compared with targets of 1 for real images and 0 for generated images; they should not be interpreted as probabilities.

### Weight Initialization and Capacity

All convolutional weights are initialized from a normal distribution with mean 0 and standard deviation 0.02. Convolutional biases are initialized to zero. A fixed PyTorch seed controls initialization for reproducible experiments within the same execution environment.

Each generator contains 1,374,723 trainable parameters, and each discriminator contains 694,241. The initial width of 32 channels and four residual blocks keep the baseline relatively small, allowing training speed and output behavior to be assessed on local hardware.

### Scope of the Baseline

This implementation follows the CycleGAN approach but does not reproduce the paper’s full experimental configuration. It uses a compact architecture and a fixed learning rate, without attention mechanisms, pretrained feature losses, or an image replay buffer. These choices establish a manageable baseline whose limitations can be examined before testing additional complexity.

The planned resolution experiment retains this architecture and changes the input size from 128 × 128 to 256 × 256. Because the networks are convolutional, their parameter counts remain unchanged, while intermediate feature maps become larger.

**Reference:** Zhu, J.-Y., Park, T., Isola, P., & Efros, A. A. (2017). *Unpaired image-to-image translation using cycle-consistent adversarial networks*. https://arxiv.org/abs/1703.10593

In [ ]:
class ResidualBlock(nn.Module):
    """Preserve spatial dimensions while learning a residual transformation.

    Parameters
    ----------
    channels : int
        Number of input and output feature channels.
    """

    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels)
        )

    def forward(self, x):
        """Transform a feature tensor.

        Parameters
        ----------
        x : torch.Tensor
            Input features of shape (N, channels, H, W).

        Returns
        -------
        torch.Tensor
            Residual output with the same shape.
        """
        return x + self.block(x)

In [ ]:
class Generator(nn.Module):
    """Translate RGB images using downsampling and residual blocks.

    Parameters
    ----------
    base_channels : int, default=32
        Feature channels in the first convolution.
    n_blocks : int, default=4
        Number of residual blocks at the bottleneck.
    """

    def __init__(self, base_channels=32, n_blocks=4):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(3, base_channels, 7),
            nn.InstanceNorm2d(base_channels),
            nn.ReLU(inplace=True)
        ]

        channels = base_channels
        for _ in range(2):
            layers.extend([
                nn.Conv2d(channels, channels * 2, 3, stride=2, padding=1),
                nn.InstanceNorm2d(channels * 2),
                nn.ReLU(inplace=True),
            ])
            channels *= 2

        # Residual transformations operate at one quarter of the input resolution.
        layers.extend(ResidualBlock(channels) for _ in range(n_blocks))

        for _ in range(2):
            layers.extend([
                # Upsample first, then learn a convolution at the larger resolution.
                nn.Upsample(scale_factor=2, mode="nearest"),
                nn.ReflectionPad2d(1),
                nn.Conv2d(channels, channels // 2, 3),
                nn.InstanceNorm2d(channels // 2),
                nn.ReLU(inplace=True)
            ])
            channels //= 2

        layers.extend([
            nn.ReflectionPad2d(3),
            nn.Conv2d(channels, 3, 7),
            nn.Tanh()
        ])
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        """Translate an image batch.

        Parameters
        ----------
        x : torch.Tensor
            Normalized RGB images of shape (N, 3, H, W);
            height and width must be divisible by four.

        Returns
        -------
        torch.Tensor
            Translated images of the same shape, bounded by [-1, 1].
        """
        return self.model(x)

In [ ]:
class Discriminator(nn.Module):
    """Score overlapping image patches as real or generated.

    Parameters
    ----------
    base_channels : int, default=32
        Feature channels in the first convolution.
    """

    def __init__(self, base_channels=32):
        super().__init__()
        layers = [
            nn.Conv2d(3, base_channels, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True)
        ]

        channels = base_channels
        for stride in [2, 2, 1]:
            next_channels = channels * 2
            layers.extend([
                nn.Conv2d(channels, next_channels, 4, stride=stride, padding=1),
                nn.InstanceNorm2d(next_channels),
                nn.LeakyReLU(0.2, inplace=True)
            ])
            channels = next_channels

        layers.append(nn.Conv2d(channels, 1, 4, padding=1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        """Score an image batch.

        Parameters
        ----------
        x : torch.Tensor
            RGB images of shape (N, 3, H, W).

        Returns
        -------
        torch.Tensor
            Unbounded patch scores of shape (N, 1, H_patch, W_patch).
        """
        return self.model(x)

In [ ]:
def initialize_weights(module):
    """Initialize convolution weights for GAN training.

    Parameters
    ----------
    module : torch.nn.Module
        Module visited by Module.apply.

    Returns
    -------
    None
        Updates convolution weights and biases in place.
    """
    if isinstance(module, nn.Conv2d):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

In [ ]:
torch.manual_seed(SEED)

G_photo_to_anime = Generator(base_channels=BASE_CHANNELS, n_blocks=RESIDUAL_BLOCKS).to(device)
G_anime_to_photo = Generator(base_channels=BASE_CHANNELS, n_blocks=RESIDUAL_BLOCKS).to(device)
D_photo = Discriminator(base_channels=BASE_CHANNELS).to(device)
D_anime = Discriminator(base_channels=BASE_CHANNELS).to(device)

models = {
    "G_photo_to_anime": G_photo_to_anime,
    "G_anime_to_photo": G_anime_to_photo,
    "D_photo": D_photo,
    "D_anime": D_anime
}

for name, model in models.items():
    model.apply(initialize_weights)
    print(f"{name}: {sum(p.numel() for p in model.parameters()):,} parameters")

with torch.no_grad():
    example = batch["photo"].to(device)
    generated = G_photo_to_anime(example)
    patch_scores = D_anime(generated)

assert generated.shape == example.shape
assert torch.isfinite(generated).all().item()
assert torch.isfinite(patch_scores).all().item()

print("\nGenerator output:", tuple(generated.shape))
print("Discriminator output:", tuple(patch_scores.shape))

## 7. Losses and Alternating Optimization

Training must balance two goals: making outputs resemble the target domain and retaining useful content from the input. The generators therefore learn from three loss terms, calculated in both translation directions.

### Generator Losses

| Loss | What it encourages | Example |
|---|---|---|
| Adversarial | Outputs that resemble real images in the target domain | A generated anime portrait should receive high realism scores from the anime discriminator |
| Cycle consistency | Recovery of the original image after translating there and back | Photo → anime → reconstructed photo |
| Identity | Minimal changes when an image already belongs to the target domain | An anime portrait passed through the photo-to-anime generator should remain similar |

**Adversarial loss** uses mean squared error between discriminator patch scores and a target of 1. During a generator update, this encourages generated images to receive scores associated with real images. The scores are not probabilities.

**Cycle-consistency loss** measures the mean absolute pixel difference, or L1 distance, between an original image and its reconstruction after translation through both generators. It is calculated for photo → anime → photo and anime → photo → anime. This discourages the generators from discarding information needed to reconstruct the input.

**Identity loss** also uses L1 distance, but compares an image with the output of the generator whose target domain it already belongs to. The photo-to-anime generator receives real anime images, and the anime-to-photo generator receives real photographs. This discourages unnecessary changes to already suitable inputs. Despite its name, identity loss does not directly measure or guarantee preservation of a person's identity.

The combined generator objective is:

$$
L_G = L_{\text{adversarial}}
      + 10L_{\text{cycle}}
      + 5L_{\text{identity}}
$$

Each term is summed across both directions before its weight is applied. The weights control the relative penalties; they are not percentages. Their actual influence also depends on the magnitude of each loss.

### Alternating Generator and Discriminator Updates

Each training step has two phases:

1. **Update both generators.** Generate translated images, calculate the three generator losses, and update generator weights. Discriminator weights are temporarily frozen. However, gradients still pass through the discriminator calculations so the generators can learn how changes to their outputs affect realism scores.

2. **Update both discriminators.** Each discriminator receives real images from its domain and generated images from the corresponding generator. Real patch scores are compared with a target of 1, and generated patch scores with a target of 0. Generated images are detached from the generator computation graph, so this phase updates only discriminator weights.

For each discriminator, the real-image and generated-image losses are averaged. The reported discriminator loss is the sum of these averages across the two domains.

### Interpreting the Training Logs

The logged `cycle` and `identity` values are the unweighted sums across both directions. The logged `generator` value includes their weights of 10 and 5.

A falling generator loss can therefore reflect better reconstruction without better anime stylization. Loss curves help diagnose training behavior, but output quality must also be assessed using fixed validation portraits.

In [ ]:
adversarial_loss = nn.MSELoss()
reconstruction_loss = nn.L1Loss()

optimizer_G = torch.optim.Adam(
    chain(G_photo_to_anime.parameters(), G_anime_to_photo.parameters()),
    lr=LEARNING_RATE,
    betas=(0.5, 0.999)
)
optimizer_D = torch.optim.Adam(
    chain(D_photo.parameters(), D_anime.parameters()),
    lr=LEARNING_RATE,
    betas=(0.5, 0.999)
)

In [ ]:
def train_step(
    batch,
    models,
    optimizer_G,
    optimizer_D,
    device,
    lambda_cycle,
    lambda_identity,
):
    """Update the supplied generators and discriminators once.

    Parameters
    ----------
    batch : dict[str, torch.Tensor]
        Normalized unpaired images under 'photo' and 'anime'.
    models : dict[str, torch.nn.Module]
        Networks keyed by 'G_photo_to_anime', 'G_anime_to_photo',
        'D_photo', and 'D_anime'.
    optimizer_G : torch.optim.Optimizer
        Optimizer for both generators.
    optimizer_D : torch.optim.Optimizer
        Optimizer for both discriminators.
    device : torch.device
        Device used for computation.
    lambda_cycle : float
        Weight applied to cycle-consistency loss.
    lambda_identity : float
        Weight applied to identity loss.

    Returns
    -------
    dict[str, float]
        Total generator and discriminator losses, plus unweighted
        adversarial, cycle, and identity losses.
    """
    G_photo_to_anime = models["G_photo_to_anime"]
    G_anime_to_photo = models["G_anime_to_photo"]
    D_photo = models["D_photo"]
    D_anime = models["D_anime"]

    mse = nn.MSELoss()
    l1 = nn.L1Loss()

    real_photo = batch["photo"].to(device)
    real_anime = batch["anime"].to(device)

    # Freeze discriminator weights while retaining the gradient path
    # through their operations to the generated images.
    for discriminator in [D_photo, D_anime]:
        discriminator.requires_grad_(False)

    optimizer_G.zero_grad(set_to_none=True)

    fake_anime = G_photo_to_anime(real_photo)
    fake_photo = G_anime_to_photo(real_anime)

    anime_scores = D_anime(fake_anime)
    photo_scores = D_photo(fake_photo)

    loss_adversarial = (
        mse(anime_scores, torch.ones_like(anime_scores))
        + mse(photo_scores, torch.ones_like(photo_scores))
    )

    # Translate there and back to reconstruct each original input.
    loss_cycle = (
        l1(G_anime_to_photo(fake_anime), real_photo)
        + l1(G_photo_to_anime(fake_photo), real_anime)
    )

    # Inputs already in the target domain should change minimally.
    loss_identity = (
        l1(G_photo_to_anime(real_anime), real_anime)
        + l1(G_anime_to_photo(real_photo), real_photo)
    )

    loss_G = (
        loss_adversarial
        + lambda_cycle * loss_cycle
        + lambda_identity * loss_identity
    )

    if not torch.isfinite(loss_G).item():
        raise RuntimeError("Non-finite generator loss.")

    loss_G.backward()
    optimizer_G.step()

    for discriminator in [D_photo, D_anime]:
        discriminator.requires_grad_(True)

    optimizer_D.zero_grad(set_to_none=True)
    loss_D = torch.zeros((), device=device)

    # Detachment ensures this phase updates only discriminator weights.
    for discriminator, real, fake in [
        (D_photo, real_photo, fake_photo),
        (D_anime, real_anime, fake_anime),
    ]:
        real_scores = discriminator(real)
        fake_scores = discriminator(fake.detach())

        loss_D = loss_D + 0.5 * (
            mse(real_scores, torch.ones_like(real_scores))
            + mse(fake_scores, torch.zeros_like(fake_scores))
        )

    if not torch.isfinite(loss_D).item():
        raise RuntimeError("Non-finite discriminator loss.")

    loss_D.backward()
    optimizer_D.step()

    return {
        "generator": loss_G.item(),
        "discriminator": loss_D.item(),
        "adversarial": loss_adversarial.item(),
        "cycle": loss_cycle.item(),
        "identity": loss_identity.item(),
    }

## 7.1. Diagnostic updates and timing

These 20 steps perform real optimizer updates and are part of the recorded training budget. They are not a read-only test. The first five timings are excluded as warm-up. The epoch estimate excludes image loading, plotting, and checkpointing. Evaluation mode skips these updates.

In [ ]:
if RUN_MODE == "train":
    for model in models.values():
        model.train()
    
    smoke_test_history = []
    step_times = []
    
    for step, training_batch in enumerate(train_loader, start=1):
        if device.type == "mps":
            torch.mps.synchronize()
    
        started = perf_counter()
        # losses = train_step(training_batch)
        losses = train_step(
            training_batch,
            models,
            optimizer_G,
            optimizer_D,
            device,
            LAMBDA_CYCLE,
            LAMBDA_IDENTITY
        )
    
        if device.type == "mps":
            torch.mps.synchronize()
    
        step_times.append(perf_counter() - started)
        smoke_test_history.append({"step": step, **losses})
    
        if step % 5 == 0:
            print(
                f"Step {step:02d} | "
                f"G: {losses['generator']:.3f} | "
                f"D: {losses['discriminator']:.3f}"
            )
    
        if step == SMOKE_STEPS:
            break
    
    seconds_per_step = float(np.median(step_times[5:]))
    
    print(f"\nMedian step time after warm-up: {seconds_per_step:.3f} seconds")
    print(
        "Estimated training compute per epoch: "
        f"{seconds_per_step * len(train_loader) / 60:.1f} minutes"
    )


## 8. Fixed validation portraits

Select eight validation photos once for progress monitoring. These never contribute gradients. They support qualitative assessment, not an unbiased test estimate.

In [ ]:
validation_photos = (
    split_manifest.loc[
        split_manifest["split"].eq("validation")
        & split_manifest["domain"].eq("photo")
    ]
    .sample(n=8, random_state=SEED)
)

fixed_images = []

for path in validation_photos["path"]:
    with Image.open(PROJECT_ROOT / path) as image:
        fixed_images.append(eval_transform(image.convert("RGB")))

fixed_validation_photos = torch.stack(fixed_images).to(device)

validation_photos.to_csv(
    PROJECT_ROOT / "data/processed/fixed_validation_photos.csv",
    index=False
)

print("Fixed validation batch:", tuple(fixed_validation_photos.shape))

## 9. Fresh baseline training and automatic checkpoints

A fresh run initializes history once and trains epochs 1–20 in order. Save epoch-specific checkpoints automatically at 10 and 20, plus an atomic latest checkpoint after every epoch.

Checkpoints include model/optimizer states, configuration, and random-number states for future resume support; this notebook exposes fresh training and evaluation modes only. Training history contains epoch averages; the 20 preceding diagnostic updates are recorded separately in checkpoint configuration. Do not rerun this training cell midway through a run; restart from section 1 for a fresh experiment.

In [ ]:
if RUN_MODE == "train":
    training_history = []
    
    for epoch in range(1, PILOT_EPOCHS + 1):
        epoch_start = time.perf_counter()
        epoch_losses = []
    
        for model in models.values():
            model.train()
    
        for training_batch in train_loader:
            epoch_losses.append(
                train_step(
                    training_batch,
                    models,
                    optimizer_G,
                    optimizer_D,
                    device,
                    LAMBDA_CYCLE,
                    LAMBDA_IDENTITY
                )
            )
    
        averages = pd.DataFrame(epoch_losses).mean().to_dict()
        training_history.append({
            "epoch": epoch,
            "minutes": (time.perf_counter() - epoch_start) / 60,
            **averages
        })
    
        # Evaluate the same validation portraits after each epoch.
        G_photo_to_anime.eval()
    
        with torch.inference_mode():
            generated = G_photo_to_anime(fixed_validation_photos)
            originals = (
                fixed_validation_photos.cpu() * 0.5 + 0.5
            ).clamp(0, 1)
            outputs = (generated.cpu() * 0.5 + 0.5).clamp(0, 1)
    
        fig, axes = plt.subplots(2, 8, figsize=(16, 5))
    
        for index in range(8):
            axes[0, index].imshow(originals[index].permute(1, 2, 0))
            axes[1, index].imshow(outputs[index].permute(1, 2, 0))
            axes[0, index].axis("off")
            axes[1, index].axis("off")
    
        axes[0, 0].set_title("Input")
        axes[1, 0].set_title("Generated")
        fig.suptitle(f"Pilot — epoch {epoch}")
        plt.tight_layout()
        fig.savefig(
            RUN_DIR / f"figures/pilot_epoch_{epoch:02d}.png",
            dpi=150,
            bbox_inches="tight"
        )
        plt.show()
        plt.close(fig)
    
        pd.DataFrame(training_history).to_csv(
            RUN_DIR / "reports/pilot_training_history.csv",
            index=False
        )
    
        checkpoint = {
            "epoch": epoch,
            "models": {
                name: model.state_dict()
                for name, model in models.items()
            },
            "optimizer_G": optimizer_G.state_dict(),
            "optimizer_D": optimizer_D.state_dict(),
            "history": training_history,
            "config": {
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE,
                "base_channels": BASE_CHANNELS,
                "residual_blocks": RESIDUAL_BLOCKS,
                "lambda_cycle": LAMBDA_CYCLE,
                "lambda_identity": LAMBDA_IDENTITY,
                "seed": SEED,
                "preceding_smoke_steps": SMOKE_STEPS,
                "learning_rate": LEARNING_RATE
            },
            "python_rng": random.getstate(),
            "numpy_rng": np.random.get_state(),
            "torch_rng": torch.get_rng_state(),
            "loader_rng": train_loader.generator.get_state()
        }
    
        if device.type == "mps":
            checkpoint["mps_rng"] = torch.mps.get_rng_state()
    
        checkpoint_path = RUN_DIR / "checkpoints/pilot_latest.pt"
        temporary_path = checkpoint_path.with_suffix(".tmp")
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(checkpoint_path)
    
        # Snapshot at the moment the named epoch completes; never relabel later weights.
        if epoch in [10, 20]:
            shutil.copy2(checkpoint_path, RUN_DIR / f"checkpoints/pilot_epoch_{epoch}.pt")
    
        print(
            f"Epoch {epoch}/{PILOT_EPOCHS} | "
            f"G: {averages['generator']:.3f} | "
            f"D: {averages['discriminator']:.3f} | "
            f"Cycle: {averages['cycle']:.3f} | "
            f"Time: {training_history[-1]['minutes']:.1f} min",
            flush=True,
        )
else:
    history_path = RUN_DIR / "reports/pilot_training_history.csv"
    training_history = pd.read_csv(history_path).to_dict("records")
    print("Loaded recorded history; no training updates performed.")


## 10. Training diagnostics

Plot adversarial and reconstruction terms separately because weighted reconstruction dominates total generator loss. These are training diagnostics, not validation scores.

In [ ]:
history_df = pd.DataFrame(training_history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for axis, columns, title in [
    (axes[0], ["generator", "discriminator"], "Total losses"),
    (axes[1], ["adversarial"], "Generator adversarial loss"),
    (axes[2], ["cycle", "identity"], "Reconstruction losses"),
]:
    history_df.plot(x="epoch", y=columns, ax=axis, marker="o")
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Fixed validation comparison: epochs 10 and 20

Select 16 additional validation portraits, excluding the eight used during training. Compare the same inputs under both checkpoints. A separate generator protects current training weights. Inspect facial structure, expression, accessories, color changes, and artifacts. Both resolution and model configuration must match the saved checkpoints. Test portraits remain untouched.

In [ ]:
comparison_rows = (
    split_manifest.loc[
        split_manifest["split"].eq("validation")
        & split_manifest["domain"].eq("photo")
        & ~split_manifest["path"].isin(validation_photos["path"])
    ]
    .sample(n=16, random_state=SEED)
    .reset_index(drop=True)
)

comparison_rows.to_csv(
    PROJECT_ROOT / "data/processed/validation_comparison_samples.csv",
    index=False,
)

comparison_images = []
for path in comparison_rows["path"]:
    with Image.open(PROJECT_ROOT / path) as image:
        comparison_images.append(eval_transform(image.convert("RGB")))

comparison_batch = torch.stack(comparison_images).to(device)
comparison_outputs = {}

# A separate model leaves the current training weights untouched.
comparison_model = Generator(base_channels=BASE_CHANNELS, n_blocks=RESIDUAL_BLOCKS).to(device)

for epoch in [10, 20]:
    checkpoint = torch.load(
        RUN_DIR / f"checkpoints/pilot_epoch_{epoch}.pt",
        map_location="cpu",
        weights_only=False  # Load only checkpoints created by this notebook.
    )
    assert checkpoint["epoch"] == epoch

    comparison_model.load_state_dict(
        checkpoint["models"]["G_photo_to_anime"]
    )
    comparison_model.eval()

    with torch.inference_mode():
        comparison_outputs[epoch] = (
            comparison_model(comparison_batch).cpu() * 0.5 + 0.5
        ).clamp(0, 1)

originals = (comparison_batch.cpu() * 0.5 + 0.5).clamp(0, 1)

for page in range(2):
    fig, axes = plt.subplots(3, 8, figsize=(16, 7))

    for column in range(8):
        index = page * 8 + column

        for row, images in enumerate([
            originals,
            comparison_outputs[10],
            comparison_outputs[20]
        ]):
            axes[row, column].imshow(
                images[index].permute(1, 2, 0),
                interpolation="nearest"
            )
            axes[row, column].axis("off")

        axes[0, column].set_title(f"Sample {index + 1}")

    fig.suptitle("Rows: input / epoch 10 / epoch 20")
    plt.tight_layout()
    fig.savefig(
        RUN_DIR / f"figures/validation_comparison_{page + 1}.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

del comparison_model, checkpoint

In [ ]:
# Preserve the completed baseline's location and settings.
# The 256-pixel experiment will save its outputs in a separate directory.
BASELINE_RUN_DIR = Path(RUN_DIR)
BASELINE_IMAGE_SIZE = IMAGE_SIZE

assert BASELINE_IMAGE_SIZE == 128
assert (
    BASELINE_RUN_DIR / "checkpoints/pilot_epoch_20.pt"
).is_file(), "The completed baseline checkpoint was not found."

print("128-pixel baseline:", BASELINE_RUN_DIR)

## 12. Resolution Experiment: 256 × 256

This experiment tests whether training at the dataset’s original resolution improves portrait detail and reduces visible artifacts.

The input resolution increases from 128 × 128 to 256 × 256. The architecture, training and validation splits, random seed, batch size, loss weights, learning rate, and training budget remain the same. Fresh generators and discriminators are trained from initialization, rather than continuing from the 128-pixel weights.

Both experiments include 20 diagnostic training updates followed by 20 epochs. Their epoch-20 outputs will be compared using the same validation portraits, assessing facial detail, expressions, accessories, texture artifacts, and unwanted color changes.

The convolutional architecture retains the same parameter count at both resolutions. However, its discriminator patches cover a smaller proportion of a 256-pixel image, so the comparison measures the effect of resolution within this fixed architecture. The test set remains reserved for final evaluation.

In [ ]:
def prepare_resolution_experiment(image_size, baseline_dir):
    """Prepare fresh models and data for a resolution experiment.

    Parameters
    ----------
    image_size : int
        Square image resolution, divisible by four.
    baseline_dir : pathlib.Path
        Completed baseline directory containing its epoch-20 checkpoint.

    Returns
    -------
    dict
        Experiment configuration, output directory, fresh networks,
        optimizers, training loader, and fixed validation tensors.
    """
    baseline_dir = Path(baseline_dir)
    if image_size < 32 or image_size % 4 != 0:
        raise ValueError("Image size must be at least 32 and divisible by four.")

    # Read the baseline settings without loading its weights into the new model.
    checkpoint = torch.load(
        baseline_dir / "checkpoints/pilot_epoch_20.pt",
        map_location="cpu",
        weights_only=False  # Trusted checkpoint created by this notebook.
    )
    assert checkpoint["epoch"] == 20
    assert checkpoint["config"]["image_size"] == 128

    config = dict(checkpoint["config"])
    config["image_size"] = image_size
    config["epochs"] = 20
    del checkpoint

    run_id = datetime.now(timezone.utc).strftime(
        f"resolution_{image_size}_%Y%m%dT%H%M%S_%fZ"
    )
    run_dir = PROJECT_ROOT / "runs" / run_id

    for folder in ["figures", "reports", "checkpoints"]:
        (run_dir / folder).mkdir(parents=True, exist_ok=True)

    # Reset sampling and augmentation before constructing the new loader.
    random.seed(config["seed"])
    np.random.seed(config["seed"])
    torch.manual_seed(config["seed"])

    training_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3)
    ])

    evaluation_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3)
    ])

    dataset = UnpairedPortraitDataset(
        split_manifest.loc[split_manifest["split"].eq("train")],
        PROJECT_ROOT,
        training_transform
    )

    loader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,
        generator=torch.Generator().manual_seed(config["seed"])
    )

    # Match the baseline's initial batch inspection and RNG consumption.
    preview_batch = next(iter(loader))

    # Reset before model creation so both resolutions start with the same
    # initialization procedure and identically shaped parameter tensors.
    torch.manual_seed(config["seed"])

    experiment_models = {
        "G_photo_to_anime": Generator(
            base_channels=config["base_channels"],
            n_blocks=config["residual_blocks"]
        ).to(device),
        "G_anime_to_photo": Generator(
            base_channels=config["base_channels"],
            n_blocks=config["residual_blocks"]
        ).to(device),
        "D_photo": Discriminator(
            base_channels=config["base_channels"]
        ).to(device),
        "D_anime": Discriminator(
            base_channels=config["base_channels"]
        ).to(device)
    }

    for model in experiment_models.values():
        model.apply(initialize_weights)

    generator_optimizer = torch.optim.Adam(
        chain(
            experiment_models["G_photo_to_anime"].parameters(),
            experiment_models["G_anime_to_photo"].parameters()
        ),
        lr=config["learning_rate"],
        betas=(0.5, 0.999)
    )

    discriminator_optimizer = torch.optim.Adam(
        chain(
            experiment_models["D_photo"].parameters(),
            experiment_models["D_anime"].parameters()
        ),
        lr=config["learning_rate"],
        betas=(0.5, 0.999)
    )

    # Reuse the exact baseline validation paths at the new resolution.
    validation_images = []
    for path in validation_photos["path"]:
        with Image.open(PROJECT_ROOT / path) as image:
            validation_images.append(
                evaluation_transform(image.convert("RGB"))
            )

    validation_batch = torch.stack(validation_images).to(device)

    # Check shapes and finite outputs without performing optimizer updates.
    with torch.no_grad():
        preview = preview_batch["photo"].to(device)
        generated = experiment_models["G_photo_to_anime"](preview)
        scores = experiment_models["D_anime"](generated)

    assert generated.shape == preview.shape
    assert torch.isfinite(generated).all().item()
    assert torch.isfinite(scores).all().item()

    # Save the data allocation alongside this experiment's outputs.
    split_manifest.to_csv(run_dir / "reports/split_manifest.csv", index=False)
    validation_photos.to_csv(run_dir / "reports/fixed_validation_photos.csv", index=False)

    print("Run directory:", run_dir)
    print("Validation batch:", tuple(validation_batch.shape))
    print("Generator output:", tuple(generated.shape))
    print("Discriminator output:", tuple(scores.shape))

    return {
        "config": config,
        "run_dir": run_dir,
        "models": experiment_models,
        "optimizer_G": generator_optimizer,
        "optimizer_D": discriminator_optimizer,
        "train_loader": loader,
        "validation_batch": validation_batch
    }

In [ ]:
experiment_256 = prepare_resolution_experiment(
    image_size=256,
    baseline_dir=BASELINE_RUN_DIR
)

In [ ]:
for model in experiment_256["models"].values():
    model.train()

diagnostic_history_256 = []
step_times_256 = []

for step, training_batch in enumerate(
    experiment_256["train_loader"], start=1
):
    # Synchronize GPU work so timing includes completed computations.
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

    started = perf_counter()

    losses = train_step(
        training_batch,
        experiment_256["models"],
        experiment_256["optimizer_G"],
        experiment_256["optimizer_D"],
        device,
        experiment_256["config"]["lambda_cycle"],
        experiment_256["config"]["lambda_identity"]
    )

    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = perf_counter() - started
    step_times_256.append(elapsed)
    diagnostic_history_256.append({
        "step": step,
        "seconds": elapsed,
        **losses
    })

    if step % 5 == 0:
        print(
            f"Step {step:02d} | "
            f"G: {losses['generator']:.3f} | "
            f"D: {losses['discriminator']:.3f}",
            flush=True
        )

    if step == experiment_256["config"]["preceding_smoke_steps"]:
        break

# Exclude the first five steps to reduce initialization effects.
seconds_per_step_256 = float(np.median(step_times_256[5:]))
minutes_per_epoch_256 = (
    seconds_per_step_256
    * len(experiment_256["train_loader"])
    / 60
)

experiment_256["diagnostic_history"] = diagnostic_history_256

pd.DataFrame(diagnostic_history_256).to_csv(
    experiment_256["run_dir"] / "reports/diagnostic_updates.csv",
    index=False
)

print(f"\nMedian step time: {seconds_per_step_256:.3f} seconds")
print(f"Estimated compute per epoch: {minutes_per_epoch_256:.1f} minutes")
print(f"Estimated compute for 20 epochs: {minutes_per_epoch_256 * 20:.1f} minutes")

In [ ]:
def train_experiment(experiment):
    """Train a prepared experiment for its configured epoch budget.

    Parameters
    ----------
    experiment : dict
        Prepared experiment containing models, optimizers, loader,
        validation tensors, configuration, and completed diagnostic history.

    Returns
    -------
    pandas.DataFrame
        Epoch-average losses and elapsed training times.
    """
    config = experiment["config"]
    run_dir = experiment["run_dir"]
    networks = experiment["models"]

    # Prevent accidentally adding another 20 epochs to an existing run.
    latest_path = run_dir / "checkpoints/pilot_latest.pt"
    if latest_path.exists():
        raise RuntimeError(
            "This run already contains a checkpoint. "
            "Prepare a new experiment or use a dedicated resume workflow."
        )

    diagnostic_history = experiment.get("diagnostic_history", [])
    if len(diagnostic_history) != config["preceding_smoke_steps"]:
        raise RuntimeError("Complete the configured diagnostic updates first.")

    history = []

    for epoch in range(1, config["epochs"] + 1):
        started = perf_counter()

        for model in networks.values():
            model.train()

        epoch_losses = []
        for training_batch in experiment["train_loader"]:
            epoch_losses.append(train_step(
                training_batch,
                networks,
                experiment["optimizer_G"],
                experiment["optimizer_D"],
                device,
                config["lambda_cycle"],
                config["lambda_identity"]
            ))

        averages = pd.DataFrame(epoch_losses).mean().to_dict()
        history.append({
            "epoch": epoch,
            "minutes": (perf_counter() - started) / 60,
            **averages
        })

        # Evaluate the same eight validation portraits after each epoch.
        generator = networks["G_photo_to_anime"]
        generator.eval()

        with torch.inference_mode():
            outputs = (
                generator(experiment["validation_batch"]).cpu() * 0.5 + 0.5
            ).clamp(0, 1)

        originals = (
            experiment["validation_batch"].cpu() * 0.5 + 0.5
        ).clamp(0, 1)

        fig, axes = plt.subplots(2, len(originals), figsize=(16, 5))
        for index in range(len(originals)):
            axes[0, index].imshow(
                originals[index].permute(1, 2, 0),
                interpolation="nearest"
            )
            axes[1, index].imshow(
                outputs[index].permute(1, 2, 0),
                interpolation="nearest"
            )
            axes[0, index].axis("off")
            axes[1, index].axis("off")

        fig.suptitle(
            f"{config['image_size']} × {config['image_size']} — "
            f"epoch {epoch} | top: input; bottom: generated"
        )
        plt.tight_layout()
        fig.savefig(
            run_dir / f"figures/pilot_epoch_{epoch:02d}.png",
            dpi=150,
            bbox_inches="tight"
        )
        plt.show()
        plt.close(fig)

        history_df = pd.DataFrame(history)
        history_df.to_csv(run_dir / "reports/pilot_training_history.csv", index=False)

        checkpoint = {
            "epoch": epoch,
            "models": {
                name: model.state_dict()
                for name, model in networks.items()
            },
            "optimizer_G": experiment["optimizer_G"].state_dict(),
            "optimizer_D": experiment["optimizer_D"].state_dict(),
            "history": history,
            "diagnostic_history": diagnostic_history,
            "config": config,
            "python_rng": random.getstate(),
            "numpy_rng": np.random.get_state(),
            "torch_rng": torch.get_rng_state(),
            "loader_rng": experiment["train_loader"].generator.get_state()
        }

        if device.type == "mps":
            checkpoint["mps_rng"] = torch.mps.get_rng_state()
        elif device.type == "cuda":
            checkpoint["cuda_rng"] = torch.cuda.get_rng_state_all()

        # Atomic replacement keeps the previous checkpoint intact until saving succeeds.
        temporary_path = latest_path.with_suffix(".tmp")
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(latest_path)

        if epoch in [10, 20]:
            shutil.copy2(latest_path, run_dir / f"checkpoints/pilot_epoch_{epoch}.pt")

        print(
            f"Epoch {epoch}/{config['epochs']} | "
            f"G: {averages['generator']:.3f} | "
            f"D: {averages['discriminator']:.3f} | "
            f"Cycle: {averages['cycle']:.3f} | "
            f"Time: {history[-1]['minutes']:.1f} min",
            flush=True
        )

    experiment["history"] = history
    return pd.DataFrame(history)

In [ ]:
history_256 = train_experiment(experiment_256)

## 13. Comparing 128 × 128 and 256 × 256 Outputs

The epoch-20 checkpoints are evaluated on the same 16 additional validation portraits used earlier. Each model receives inputs at its training resolution.

Outputs are displayed in equally sized panels using nearest-neighbor interpolation, making the lower resolution visible without introducing
smoothing. Compare facial detail, texture artifacts, expressions, accessories, and unwanted color changes. The test set remains untouched.

In [ ]:
resolution_outputs = {}

run_directories = {
    128: BASELINE_RUN_DIR,
    256: experiment_256["run_dir"],
}

for resolution, run_directory in run_directories.items():
    checkpoint = torch.load(
        run_directory / "checkpoints/pilot_epoch_20.pt",
        map_location="cpu",
        weights_only=False,  # Trusted checkpoints created by this notebook.
    )
    config = checkpoint["config"]

    assert checkpoint["epoch"] == 20
    assert config["image_size"] == resolution

    # Separate evaluation models leave the training networks unchanged.
    evaluation_model = Generator(
        base_channels=config["base_channels"],
        n_blocks=config["residual_blocks"],
    ).to(device)

    evaluation_model.load_state_dict(
        checkpoint["models"]["G_photo_to_anime"]
    )
    evaluation_model.eval()

    evaluation_transform = transforms.Compose([
        transforms.Resize((resolution, resolution)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3),
    ])

    generated_images = []

    # Process one image at a time to keep evaluation memory modest.
    with torch.inference_mode():
        for path in comparison_rows["path"]:
            with Image.open(PROJECT_ROOT / path) as image:
                tensor = evaluation_transform(
                    image.convert("RGB")
                ).unsqueeze(0).to(device)

            generated = evaluation_model(tensor)[0].cpu()
            generated_images.append(
                (generated * 0.5 + 0.5).clamp(0, 1)
            )

    resolution_outputs[resolution] = generated_images
    del evaluation_model, checkpoint

comparison_rows.to_csv(
    experiment_256["run_dir"] / "reports/resolution_comparison_samples.csv",
    index=False,
)

for page in range(2):
    fig, axes = plt.subplots(3, 8, figsize=(18, 8))

    for column in range(8):
        index = page * 8 + column
        path = comparison_rows.iloc[index]["path"]

        with Image.open(PROJECT_ROOT / path) as image:
            axes[0, column].imshow(
                image.convert("RGB"),
                interpolation="nearest",
            )

        for row, resolution in [(1, 128), (2, 256)]:
            axes[row, column].imshow(
                resolution_outputs[resolution][index].permute(1, 2, 0),
                interpolation="nearest",
            )

        axes[0, column].set_title(f"Sample {index + 1}")

        for row in range(3):
            axes[row, column].axis("off")

    fig.suptitle(
        "Epoch 20 comparison — rows: original / 128 × 128 / 256 × 256"
    )
    plt.tight_layout()
    fig.savefig(
        experiment_256["run_dir"]
        / f"figures/resolution_comparison_{page + 1}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

## 14. Attention-based portrait translation: experiment design

The resolution comparison did not show a consistent improvement from increasing the compact CycleGAN’s input size from 128 × 128 to 256 × 256. Higher-resolution outputs contained finer detail, but several portraits also developed fragmented shading and distorted facial features. These observations apply to the inspected validation examples from one training run at each resolution.

The next experiment changes the model design. It draws on U-GAT-IT, which combines attention with adaptive normalization for unpaired image translation ([Kim et al., 2020](https://arxiv.org/abs/1907.10830)).

### What changes?

**Attention** gives the model a learned way to emphasize features useful for distinguishing the photo and anime domains. It does not use manually annotated eyes, mouths, or face regions, and it does not guarantee anatomically correct outputs.

**Adaptive layer-instance normalization (AdaLIN)** learns how to combine instance and layer normalization. Its affine parameters are conditioned on features extracted from the input image, allowing the generator to adjust its feature transformation for each portrait.

The implementation will also introduce additional architectural and training changes. Consequently, this is a comparison between model configurations rather than an experiment that isolates the effect of attention alone.

### Implementation and compute checks

The model will be trained from scratch using PyTorch. Before a longer training run, a short diagnostic will check tensor shapes, finite losses, gradient flow, memory use where measurable, and training speed on the local Apple Silicon GPU.

The exact channel counts, residual blocks, discriminator structure, loss weights, and training schedule will be recorded with the run. Any departures from the published architecture or training procedure will be documented. A reduced configuration will be described as U-GAT-IT-inspired rather than a reproduction of the paper.

### Evaluation

The completed CycleGAN runs remain the baselines. Progress images will use the same fixed validation portraits, and the final comparison will reuse the same 16 additional validation portraits.

Evaluation will consider:

- Coherence of eyes, mouths, facial contours, and hair.
- Strength and consistency of anime styling.
- Preservation of pose, expression, skin tone, and recognizable characteristics.
- Visible artifacts and failures across different portraits.
- Training time and computational cost.

Training losses will support diagnosis, but lower losses alone will not establish better visual quality. The test set remains untouched during model development and selection.

### Reference

Kim, J., Kim, M., Kang, H., & Lee, K. (2020).
*U-GAT-IT: Unsupervised generative attentional networks with adaptive
layer-instance normalization for image-to-image translation*. International
Conference on Learning Representations.
https://arxiv.org/abs/1907.10830

### 14.1. Normalization layers

The generator will use two normalization layers inspired by [Kim et al. (2020)](https://arxiv.org/abs/1907.10830):

- **AdaLIN** combines instance normalization, computed separately for each feature channel, with layer normalization, computed across all channels and spatial positions within an image. The generator supplies input-dependent scale and offset parameters.
- **LIN** uses the same normalization mixture but learns fixed scale and offset parameters. It will be used during upsampling.

A learned parameter, rho, controls the mixture for each channel. A value of 1 selects instance normalization; 0 selects layer normalization. Training will constrain rho to [0, 1] after each generator update.

This implementation uses population variance (`unbiased=False`) consistently for both normalization calculations. A small epsilon prevents division by zero.

In [ ]:
class LayerInstanceNorm(nn.Module):
    """Combine instance and layer normalization for image features.

    Parameters
    ----------
    channels : int
        Number of feature channels.
    adaptive : bool, default=False
        If True, receive scale and offset from the generator.
        Otherwise, learn fixed scale and offset parameters.
    rho_init : float, default=0.0
        Initial instance-normalization weight, between zero and one.
    eps : float, default=1e-5
        Numerical stability constant.
    """

    def __init__(self, channels, adaptive=False, rho_init=0.0, eps=1e-5):
        super().__init__()

        if not 0.0 <= rho_init <= 1.0:
            raise ValueError("rho_init must lie in [0, 1].")

        self.adaptive = adaptive
        self.eps = eps

        # Each feature channel learns its own normalization mixture.
        self.rho = nn.Parameter(
            torch.full((1, channels, 1, 1), float(rho_init))
        )

        if not adaptive:
            self.gamma = nn.Parameter(torch.ones(1, channels, 1, 1))
            self.beta = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, x, gamma=None, beta=None):
        """Normalize features and apply scale and offset.

        Parameters
        ----------
        x : torch.Tensor
            Feature tensor with shape (batch, channels, height, width).
        gamma : torch.Tensor or None
            Adaptive scale with shape (batch, channels).
        beta : torch.Tensor or None
            Adaptive offset with shape (batch, channels).

        Returns
        -------
        torch.Tensor
            Normalized features with the same shape as x.
        """
        # Instance statistics: one mean and variance per image and channel.
        instance_var, instance_mean = torch.var_mean(x, dim=(2, 3), keepdim=True, unbiased=False)
        instance_normalized = ((x - instance_mean) * torch.rsqrt(instance_var + self.eps))

        # Layer statistics: one mean and variance across each entire image.
        layer_var, layer_mean = torch.var_mean(x, dim=(1, 2, 3), keepdim=True, unbiased=False)
        layer_normalized = ((x - layer_mean) * torch.rsqrt(layer_var + self.eps))

        normalized = (self.rho * instance_normalized + (1.0 - self.rho) * layer_normalized)

        if self.adaptive:
            expected_shape = (x.shape[0], x.shape[1])

            if gamma is None or beta is None:
                raise ValueError("Adaptive normalization requires gamma and beta.")
            if gamma.shape != expected_shape or beta.shape != expected_shape:
                raise ValueError(
                    f"gamma and beta must both have shape {expected_shape}."
                )

            # Broadcast each image's channel parameters over its spatial axes.
            scale = gamma[:, :, None, None]
            offset = beta[:, :, None, None]
        else:
            scale = self.gamma
            offset = self.beta

        return normalized * scale + offset

    @torch.no_grad()
    def constrain_rho(self):
        """Keep the learned normalization mixture within [0, 1].

        Returns
        -------
        None
            Updates rho in place.
        """
        self.rho.clamp_(0.0, 1.0)

### 14.2. Attention-based generator

The generator first reduces the image’s spatial resolution and processes its features through residual blocks.

Its attention module uses both global average pooling and global maximum pooling. Two learned classifiers produce attention logits, while their weights reweight the feature channels. The resulting features support both image generation and an attention heatmap.

A small network converts the pooled attended features into the scale and offset parameters used by AdaLIN residual blocks. Upsampling layers then reconstruct the output image using LIN and nearest-neighbor interpolation followed by convolution.

This configuration uses a pooled conditioning network, avoiding a large fully connected layer tied to the image resolution. Channel counts and residual-block counts remain configurable.

The generator returns three tensors:

1. A translated RGB image normalized to [-1, 1].
2. Two attention logits for the generator's attention-classification loss.
3. A spatial heatmap for inspecting its learned attention.

The heatmap is a diagnostic visualization, not a face segmentation mask or a guarantee that important facial features are preserved.

In [ ]:
import torch.nn.functional as F

In [ ]:
class AdaptiveResidualBlock(nn.Module):
    """Transform features using two input-conditioned AdaLIN layers.

    Parameters
    ----------
    channels : int
        Number of input and output feature channels.
    """

    def __init__(self, channels):
        super().__init__()
        self.pad = nn.ReflectionPad2d(1)
        self.conv1 = nn.Conv2d(channels, channels, 3, bias=False)
        self.conv2 = nn.Conv2d(channels, channels, 3, bias=False)

        # Start these layers close to instance normalization.
        self.norm1 = LayerInstanceNorm(channels, adaptive=True, rho_init=0.9)
        self.norm2 = LayerInstanceNorm(channels, adaptive=True, rho_init=0.9)

    def forward(self, x, gamma, beta):
        """Apply an adaptive residual transformation.

        Parameters
        ----------
        x : torch.Tensor
            Features shaped (batch, channels, height, width).
        gamma : torch.Tensor
            Input-dependent scale shaped (batch, channels).
        beta : torch.Tensor
            Input-dependent offset shaped (batch, channels).

        Returns
        -------
        torch.Tensor
            Transformed features with the same shape as x.
        """
        residual = self.conv1(self.pad(x))
        residual = F.relu(self.norm1(residual, gamma, beta))
        residual = self.conv2(self.pad(residual))
        residual = self.norm2(residual, gamma, beta)
        return x + residual

In [ ]:
class AttentionGenerator(nn.Module):
    """Translate portraits using channel attention and adaptive normalization.

    Parameters
    ----------
    base_channels : int, default=32
        Number of channels in the first convolution.
    n_blocks : int, default=4
        Number of standard residual blocks and, separately, AdaLIN blocks.
        The generator therefore contains twice this many residual blocks.
    """

    def __init__(self, base_channels=32, n_blocks=4):
        super().__init__()

        encoder = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(3, base_channels, 7, bias=False),
            nn.InstanceNorm2d(base_channels),
            nn.ReLU()
        ]

        channels = base_channels
        for _ in range(2):
            encoder.extend([
                nn.ReflectionPad2d(1),
                nn.Conv2d(channels, channels * 2, 3, stride=2, bias=False),
                nn.InstanceNorm2d(channels * 2),
                nn.ReLU()
            ])
            channels *= 2

        # Reuse the standard residual block defined in Section 6.
        encoder.extend(ResidualBlock(channels) for _ in range(n_blocks))
        self.encoder = nn.Sequential(*encoder)

        # Average and maximum pooling provide complementary channel summaries.
        self.gap_classifier = nn.Linear(channels, 1, bias=False)
        self.gmp_classifier = nn.Linear(channels, 1, bias=False)
        self.attention_fusion = nn.Conv2d(channels * 2, channels, 1)

        # Pooled conditioning keeps this network independent of image size.
        self.conditioning = nn.Sequential(
            nn.Linear(channels, channels, bias=False),
            nn.ReLU(),
            nn.Linear(channels, channels, bias=False),
            nn.ReLU(),
        )
        self.gamma = nn.Linear(channels, channels, bias=False)
        self.beta = nn.Linear(channels, channels, bias=False)

        self.adaptive_blocks = nn.ModuleList([
            AdaptiveResidualBlock(channels)
            for _ in range(n_blocks)
        ])

        decoder = []
        for _ in range(2):
            decoder.extend([
                nn.Upsample(scale_factor=2, mode="nearest"),
                nn.ReflectionPad2d(1),
                nn.Conv2d(channels, channels // 2, 3, bias=False),
                # Upsampling starts with layer normalization.
                LayerInstanceNorm(channels // 2, rho_init=0.0),
                nn.ReLU()
            ])
            channels //= 2

        decoder.extend([
            nn.ReflectionPad2d(3),
            nn.Conv2d(channels, 3, 7, bias=False),
            nn.Tanh()
        ])
        self.decoder = nn.Sequential(*decoder)

    def forward(self, x):
        """Generate a translated image and attention diagnostics.

        Parameters
        ----------
        x : torch.Tensor
            Normalized RGB images shaped (batch, 3, height, width).
            Height and width must be divisible by four.

        Returns
        -------
        tuple[torch.Tensor, torch.Tensor, torch.Tensor]
            Translated images, attention logits shaped (batch, 2),
            and heatmaps shaped (batch, 1, height/4, width/4).
        """
        if x.ndim != 4 or x.shape[1] != 3:
            raise ValueError("Expected images shaped (batch, 3, height, width).")
        if x.shape[2] % 4 or x.shape[3] % 4:
            raise ValueError("Image height and width must be divisible by four.")

        features = self.encoder(x)

        average = F.adaptive_avg_pool2d(features, 1).flatten(1)
        maximum = F.adaptive_max_pool2d(features, 1).flatten(1)

        attention_logits = torch.cat([
            self.gap_classifier(average),
            self.gmp_classifier(maximum)
        ], dim=1)

        # Classifier weights become learned channel-attention weights.
        average_features = features * self.gap_classifier.weight[
            :, :, None, None
        ]
        maximum_features = features * self.gmp_classifier.weight[
            :, :, None, None
        ]

        attended = F.relu(self.attention_fusion(
            torch.cat([average_features, maximum_features], dim=1)
        ))
        heatmap = attended.sum(dim=1, keepdim=True)

        pooled = F.adaptive_avg_pool2d(attended, 1).flatten(1)
        conditioning = self.conditioning(pooled)
        gamma = self.gamma(conditioning)
        beta = self.beta(conditioning)

        translated = attended
        for block in self.adaptive_blocks:
            translated = block(translated, gamma, beta)

        return self.decoder(translated), attention_logits, heatmap

### 14.3. Attention-based discriminators

Each image domain will have two discriminators:

- A local discriminator evaluates smaller image regions.
- A deeper global discriminator has a larger receptive field and evaluates broader image structure.

These names describe their relative receptive fields. Both return grids of patch scores rather than a single whole-image probability.

Each discriminator also uses average-pooled and maximum-pooled features to produce attention logits and reweight feature channels. Training will supervise both the patch scores and the attention logits.

Spectral normalization is applied to discriminator convolution and linear weights. This controls their spectral norm during training; it does not guarantee stable training or good generated images.

The implementation caps feature width at 512 channels to limit computational cost. The selected depths and widths will be recorded as part of this U-GAT-IT-inspired configuration.

In [ ]:
from torch.nn.utils import spectral_norm

In [ ]:
class AttentionDiscriminator(nn.Module):
    """Evaluate image patches using spectrally normalized attention features.

    Parameters
    ----------
    base_channels : int, default=32
        Number of channels in the first convolution.
    n_layers : int, default=4
        Convolutional depth, including the final patch-scoring layer.
        Larger values increase downsampling and receptive-field size.
    max_channels : int, default=512
        Maximum feature width.
    """

    def __init__(self, base_channels=32, n_layers=4, max_channels=512):
        super().__init__()

        if n_layers < 3:
            raise ValueError("n_layers must be at least 3.")
        if not 0 < base_channels <= max_channels:
            raise ValueError(
                "Require 0 < base_channels <= max_channels."
            )

        layers = [
            nn.ReflectionPad2d(1),
            spectral_norm(nn.Conv2d(3, base_channels, 4, stride=2)),
            nn.LeakyReLU(0.2)
        ]

        channels = base_channels

        # Additional stride-two layers increase the receptive field.
        for _ in range(n_layers - 3):
            next_channels = min(channels * 2, max_channels)
            layers.extend([
                nn.ReflectionPad2d(1),
                spectral_norm(
                    nn.Conv2d(channels, next_channels, 4, stride=2)
                ),
                nn.LeakyReLU(0.2)
            ])
            channels = next_channels

        next_channels = min(channels * 2, max_channels)
        layers.extend([
            nn.ReflectionPad2d(1),
            spectral_norm(nn.Conv2d(channels, next_channels, 4, stride=1)),
            nn.LeakyReLU(0.2)
        ])
        channels = next_channels
        self.features = nn.Sequential(*layers)

        self.gap_classifier = spectral_norm(
            nn.Linear(channels, 1, bias=False)
        )
        self.gmp_classifier = spectral_norm(
            nn.Linear(channels, 1, bias=False)
        )

        self.attention_fusion = spectral_norm(
            nn.Conv2d(channels * 2, channels, 1)
        )
        self.patch_head = nn.Sequential(
            nn.ReflectionPad2d(1),
            spectral_norm(
                nn.Conv2d(channels, 1, 4, stride=1, bias=False)
            )
        )

    def forward(self, x):
        """Return patch scores, attention logits, and a spatial heatmap.

        Parameters
        ----------
        x : torch.Tensor
            Normalized RGB images shaped (batch, 3, height, width).
            Spatial dimensions must support the configured depth.

        Returns
        -------
        tuple[torch.Tensor, torch.Tensor, torch.Tensor]
            Unbounded patch scores, attention logits shaped (batch, 2),
            and an attention heatmap at the feature-map resolution.
        """
        features = self.features(x)

        average = F.adaptive_avg_pool2d(features, 1).flatten(1)
        maximum = F.adaptive_max_pool2d(features, 1).flatten(1)

        # Calling these layers also updates spectral-normalization estimates
        # when the discriminator is in training mode.
        average_logit = self.gap_classifier(average)
        maximum_logit = self.gmp_classifier(maximum)
        attention_logits = torch.cat([average_logit, maximum_logit], dim=1)

        average_features = features * self.gap_classifier.weight[
            :, :, None, None
        ]
        maximum_features = features * self.gmp_classifier.weight[
            :, :, None, None
        ]

        attended = F.leaky_relu(
            self.attention_fusion(
                torch.cat([average_features, maximum_features], dim=1)
            ),
            negative_slope=0.2
        )

        heatmap = attended.sum(dim=1, keepdim=True)
        patch_scores = self.patch_head(attended)

        return patch_scores, attention_logits, heatmap

In [ ]:
# Check architecture wiring before allocating the full experiment on the GPU.
# fork_rng preserves the CPU random-number state used elsewhere in the notebook.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(SEED)

    probe_generator = AttentionGenerator(base_channels=32, n_blocks=4).eval()

    probe_discriminators = {
        "local": AttentionDiscriminator(base_channels=32, n_layers=4).eval(),
        "global": AttentionDiscriminator(base_channels=32, n_layers=6).eval()
    }

    probe_input = torch.randn(1, 3, 256, 256)

    with torch.inference_mode():
        probe_output, probe_logits, probe_heatmap = (
            probe_generator(probe_input)
        )

        assert probe_output.shape == probe_input.shape
        assert probe_logits.shape == (1, 2)
        assert probe_heatmap.shape == (1, 1, 64, 64)

        for tensor in (probe_output, probe_logits, probe_heatmap):
            assert torch.isfinite(tensor).all()

        assert probe_output.min() >= -1
        assert probe_output.max() <= 1

        print("Generator parameters:",
              f"{sum(p.numel() for p in probe_generator.parameters()):,}")
        print("Generator output:", tuple(probe_output.shape))
        print("Generator attention logits:", tuple(probe_logits.shape))
        print("Generator heatmap:", tuple(probe_heatmap.shape))

        for name, discriminator in probe_discriminators.items():
            scores, logits, heatmap = discriminator(probe_output)

            assert logits.shape == (1, 2)
            for tensor in (scores, logits, heatmap):
                assert torch.isfinite(tensor).all()

            print(f"\n{name.capitalize()} discriminator parameters:",
                  f"{sum(p.numel() for p in discriminator.parameters()):,}")
            print("Patch scores:", tuple(scores.shape))
            print("Attention logits:", tuple(logits.shape))
            print("Heatmap:", tuple(heatmap.shape))

# Release temporary models and tensors after the check.
del (
    probe_generator,
    probe_discriminators,
    probe_input,
    probe_output,
    probe_logits,
    probe_heatmap,
    discriminator,
    scores,
    logits,
    heatmap
)

print("\nArchitecture shape checks passed.")

In [ ]:
# Use a local variable so existing experiment settings remain unchanged.
probe_device = torch.device(device)
print("Checking gradient flow on:", probe_device)

with torch.random.fork_rng(devices=[]):
    torch.manual_seed(SEED)

    probe_generator = AttentionGenerator(base_channels=32, n_blocks=4).to(probe_device).train()

    probe_discriminator = AttentionDiscriminator(base_channels=32, n_layers=6).to(probe_device).train()

    # Deterministic synthetic input avoids consuming the GPU random stream.
    probe_input = torch.linspace(
        -1, 1, steps=3 * 256 * 256
    ).reshape(1, 3, 256, 256).to(probe_device)

    generated, generator_logits, _ = probe_generator(probe_input)
    scores, discriminator_logits, _ = probe_discriminator(generated)

    # Exercise image synthesis and both attention-classification branches.
    # This diagnostic objective is not the experiment's training objective.
    probe_loss = (
        generated.square().mean()
        + (scores - 1).square().mean()
        + generator_logits.square().mean()
        + discriminator_logits.square().mean()
    )

    assert torch.isfinite(probe_loss), "Non-finite diagnostic loss."
    probe_loss.backward()

    for name, network in [
        ("Generator", probe_generator),
        ("Global discriminator", probe_discriminator),
    ]:
        missing = []
        nonfinite = []
        gradient_total = 0.0

        for parameter_name, parameter in network.named_parameters():
            if not parameter.requires_grad:
                continue
            if parameter.grad is None:
                missing.append(parameter_name)
            elif not torch.isfinite(parameter.grad).all().item():
                nonfinite.append(parameter_name)
            else:
                gradient_total += parameter.grad.abs().sum().item()

        assert not missing, f"{name}: missing gradients: {missing}"
        assert not nonfinite, f"{name}: non-finite gradients: {nonfinite}"
        assert gradient_total > 0, f"{name}: all gradients are zero."

        print(f"{name}: finite gradients; total magnitude={gradient_total:.3f}")

    # Verify the constraint that will follow each generator optimizer update.
    for module in probe_generator.modules():
        if isinstance(module, LayerInstanceNorm):
            module.constrain_rho()
            assert ((module.rho >= 0) & (module.rho <= 1)).all().item()

    print("Diagnostic loss:", probe_loss.detach().item())

del (
    probe_generator,
    probe_discriminator,
    probe_input,
    generated,
    generator_logits,
    scores,
    discriminator_logits,
    probe_loss,
    network,
    parameter,
    module,
    _
)

if probe_device.type == "mps":
    torch.mps.empty_cache()

print("Gradient and normalization-constraint checks passed.")

### 14.4. Experiment configuration and preparation

This experiment uses 256 × 256 images, two generators, and four discriminators: one local and one global discriminator per domain.

Each generator has four standard residual blocks followed by four AdaLIN residual blocks. Training uses batch size one and Adam with an initial learning rate of 0.0001.

The planned loss weights are:

| Generator loss term | Weight |
|---|---:|
| Adversarial, including discriminator attention scores | 1 |
| Cycle consistency | 10 |
| Identity reconstruction | 10 |
| Generator attention classification | 1,000 |

These weights apply to the specific loss definitions introduced next. Their magnitudes cannot be interpreted independently of those definitions.

The first stage is a 20-epoch pilot with a constant learning rate, preceded by 20 recorded diagnostic updates. This is an initial development budget rather than a reproduction of the published training schedule. Timing and validation outputs will guide subsequent training decisions.

Training and validation partitions remain unchanged. Outputs, configuration, and manifests are saved in a separate experiment directory.

In [ ]:
import json


def initialize_attention_weights(module):
    """Initialize convolution and linear weights, including spectral weights.

    Parameters
    ----------
    module : torch.nn.Module
        Module visited by Module.apply.

    Returns
    -------
    None
        Initializes weights and biases in place, preserving normalization
        parameters such as rho, gamma, and beta.
    """
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        # Spectral normalization stores the trainable weight as weight_orig.
        weight = getattr(module, "weight_orig", module.weight)
        nn.init.normal_(weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)


attention_config = {
    "architecture": "ugat it inspired pooled conditioning",
    "image_size": 256,
    "batch_size": 1,
    "base_channels": 32,
    "residual_blocks_per_stage": 4,
    "local_discriminator_layers": 4,
    "global_discriminator_layers": 6,
    "max_discriminator_channels": 512,
    "learning_rate": 0.0001,
    "adam_betas": [0.5, 0.999],
    "lambda_adversarial": 1.0,
    "lambda_cycle": 10.0,
    "lambda_identity": 10.0,
    "lambda_cam": 1000.0,
    "epochs": 20,
    "diagnostic_steps": 20,
    "lr_schedule": "constant during initial pilot",
    "seed": SEED
}

random.seed(attention_config["seed"])
np.random.seed(attention_config["seed"])
torch.manual_seed(attention_config["seed"])

attention_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3)
])
attention_eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3)
])

attention_dataset = UnpairedPortraitDataset(
    split_manifest.loc[split_manifest["split"].eq("train")],
    PROJECT_ROOT,
    attention_train_transform
)
attention_loader = DataLoader(
    attention_dataset,
    batch_size=attention_config["batch_size"],
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(attention_config["seed"])
)

attention_models = {
    "G_photo_to_anime": AttentionGenerator(32, 4),
    "G_anime_to_photo": AttentionGenerator(32, 4)
}
for domain in ("photo", "anime"):
    for scale, depth in (("local", 4), ("global", 6)):
        attention_models[f"D_{domain}_{scale}"] = (
            AttentionDiscriminator(
                base_channels=32,
                n_layers=depth,
                max_channels=512
            )
        )

for name, network in attention_models.items():
    network.apply(initialize_attention_weights)
    network.to(device)
    print(f"{name}: {sum(p.numel() for p in network.parameters()):,} parameters")

attention_optimizer_G = torch.optim.Adam(
    chain(
        attention_models["G_photo_to_anime"].parameters(),
        attention_models["G_anime_to_photo"].parameters()
    ),
    lr=attention_config["learning_rate"],
    betas=tuple(attention_config["adam_betas"])
)
attention_optimizer_D = torch.optim.Adam(
    chain.from_iterable(
        network.parameters()
        for name, network in attention_models.items()
        if name.startswith("D_")
    ),
    lr=attention_config["learning_rate"],
    betas=tuple(attention_config["adam_betas"])
)

# Preserve the same eight validation portraits used by the baselines.
attention_validation_images = []
for path in validation_photos["path"]:
    with Image.open(PROJECT_ROOT / path) as image:
        attention_validation_images.append(
            attention_eval_transform(image.convert("RGB"))
        )

attention_validation_batch = torch.stack(attention_validation_images).to(device)

attention_run_id = datetime.now(timezone.utc).strftime("attention_256_%Y%m%dT%H%M%S_%fZ")
attention_run_dir = PROJECT_ROOT / "runs" / attention_run_id

for folder in ("figures", "reports", "checkpoints"):
    (attention_run_dir / folder).mkdir(parents=True, exist_ok=False)

with (attention_run_dir / "reports/config.json").open("w") as handle:
    json.dump(attention_config, handle, indent=2)

split_manifest.to_csv(
    attention_run_dir / "reports/split_manifest.csv", index=False
)
validation_photos.to_csv(
    attention_run_dir / "reports/fixed_validation_photos.csv", index=False
)

attention_experiment = {
    "config": attention_config,
    "run_dir": attention_run_dir,
    "models": attention_models,
    "optimizer_G": attention_optimizer_G,
    "optimizer_D": attention_optimizer_D,
    "train_loader": attention_loader,
    "validation_batch": attention_validation_batch,
    "diagnostic_history": []
}

print("\nRun directory:", attention_run_dir)
print("Validation batch:", tuple(attention_validation_batch.shape))
print("Prepared fresh models; no training updates performed.")

### 14.5. Losses and alternating optimization

Each training step updates the four discriminators first, followed by the two generators.

#### Discriminator objective

Each local and global discriminator produces patch scores and two attention logits. Least-squares losses train both outputs toward 1 for real images and 0 for generated images. Each discriminator contributes half the sum of its real and generated losses; these contributions are summed across both scales and domains.

Generated images are created without generator gradients during this update.

#### Generator objective

The generator objective combines four terms:

- **Adversarial:** generated images should receive discriminator patch scores and attention logits near 1, at both scales.
- **Cycle consistency:** translating to the other domain and back should reconstruct the input.
- **Identity reconstruction:** an image already in the output domain should change as little as possible.
- **Generator attention classification:** each generator's attention logits distinguish source-domain inputs from target-domain inputs using binary cross-entropy.

For the photo-to-anime generator, photo inputs receive attention labels of 1 and anime inputs receive labels of 0. The reverse generator uses the opposite domain assignment. These are domain labels, not facial-identity labels.

The total generator loss is:

**adversarial + 10 × cycle + 10 × identity + 1,000 × attention classification**

Cycle, identity, and generator attention-classification losses are summed over both translation directions.

During the generator update, discriminator parameters are frozen and the discriminators use evaluation mode. Gradients still pass through them to the generators, while spectral-normalization estimates remain fixed. After the generator update, normalization mixture parameters are constrained to [0, 1].

Reported component losses are unweighted. Total losses are not directly comparable with the compact CycleGAN because the objectives and number of discriminators differ.

In [ ]:
def attention_train_step(batch, experiment, device):
    """Update attention discriminators and generators once.

    Parameters
    ----------
    batch : dict[str, torch.Tensor]
        Unpaired normalized images under 'photo' and 'anime'.
    experiment : dict
        Prepared attention experiment containing models, optimizers,
        and loss-weight configuration.
    device : torch.device
        Device used for training.

    Returns
    -------
    dict[str, float]
        Total generator/discriminator losses and unweighted generator
        adversarial, cycle, identity, and attention-classification losses.
    """
    networks = experiment["models"]
    config = experiment["config"]
    optimizer_G = experiment["optimizer_G"]
    optimizer_D = experiment["optimizer_D"]

    G_pa = networks["G_photo_to_anime"]
    G_ap = networks["G_anime_to_photo"]
    discriminators = {
        domain: [
            networks[f"D_{domain}_local"],
            networks[f"D_{domain}_global"]
        ]
        for domain in ("photo", "anime")
    }

    real_photo = batch["photo"].to(device)
    real_anime = batch["anime"].to(device)

    mse = nn.MSELoss()
    l1 = nn.L1Loss()
    bce = nn.BCEWithLogitsLoss()

    G_pa.train()
    G_ap.train()

    # Clear gradients from the preceding step before either update.
    optimizer_G.zero_grad(set_to_none=True)
    optimizer_D.zero_grad(set_to_none=True)

    for domain_networks in discriminators.values():
        for discriminator in domain_networks:
            discriminator.requires_grad_(True)
            discriminator.train()

    # Discriminator update: generator operations need no gradient graph.
    with torch.no_grad():
        fake_anime, _, _ = G_pa(real_photo)
        fake_photo, _, _ = G_ap(real_anime)

    loss_D = real_photo.new_zeros(())

    for domain, real, fake in [
        ("photo", real_photo, fake_photo),
        ("anime", real_anime, fake_anime),
    ]:
        for discriminator in discriminators[domain]:
            real_scores, real_cam, _ = discriminator(real)
            fake_scores, fake_cam, _ = discriminator(fake)

            loss_D = loss_D + 0.5 * (
                mse(real_scores, torch.ones_like(real_scores))
                + mse(fake_scores, torch.zeros_like(fake_scores))
                + mse(real_cam, torch.ones_like(real_cam))
                + mse(fake_cam, torch.zeros_like(fake_cam))
            )

    if not torch.isfinite(loss_D).item():
        raise FloatingPointError("Non-finite discriminator loss.")

    loss_D.backward()
    optimizer_D.step()
    optimizer_D.zero_grad(set_to_none=True)

    # Freeze discriminator weights and spectral-normalization estimates.
    for domain_networks in discriminators.values():
        for discriminator in domain_networks:
            discriminator.requires_grad_(False)
            discriminator.eval()

    try:
        # Build fresh generator graphs after the discriminator update.
        fake_anime, source_cam_pa, _ = G_pa(real_photo)
        fake_photo, source_cam_ap, _ = G_ap(real_anime)

        recovered_photo, _, _ = G_ap(fake_anime)
        recovered_anime, _, _ = G_pa(fake_photo)

        identity_anime, target_cam_pa, _ = G_pa(real_anime)
        identity_photo, target_cam_ap, _ = G_ap(real_photo)

        loss_adversarial = real_photo.new_zeros(())

        for domain, fake in [("photo", fake_photo), ("anime", fake_anime)]:
            for discriminator in discriminators[domain]:
                scores, cam, _ = discriminator(fake)
                loss_adversarial = loss_adversarial + (
                    mse(scores, torch.ones_like(scores))
                    + mse(cam, torch.ones_like(cam))
                )

        loss_cycle = (
            l1(recovered_photo, real_photo)
            + l1(recovered_anime, real_anime)
        )
        loss_identity = (
            l1(identity_photo, real_photo)
            + l1(identity_anime, real_anime)
        )

        # Each generator distinguishes its source domain from its target.
        loss_cam = (
            bce(source_cam_pa, torch.ones_like(source_cam_pa))
            + bce(target_cam_pa, torch.zeros_like(target_cam_pa))
            + bce(source_cam_ap, torch.ones_like(source_cam_ap))
            + bce(target_cam_ap, torch.zeros_like(target_cam_ap))
        )

        loss_G = (
            config["lambda_adversarial"] * loss_adversarial
            + config["lambda_cycle"] * loss_cycle
            + config["lambda_identity"] * loss_identity
            + config["lambda_cam"] * loss_cam
        )

        if not torch.isfinite(loss_G).item():
            raise FloatingPointError("Non-finite generator loss.")

        loss_G.backward()
        optimizer_G.step()

        # Project learned normalization mixtures back into their valid range.
        for generator in (G_pa, G_ap):
            for module in generator.modules():
                if isinstance(module, LayerInstanceNorm):
                    module.constrain_rho()

    finally:
        # Restore discriminator state even if the generator phase raises.
        for domain_networks in discriminators.values():
            for discriminator in domain_networks:
                discriminator.requires_grad_(True)
                discriminator.train()

    return {
        "generator": loss_G.detach().item(),
        "discriminator": loss_D.detach().item(),
        "adversarial": loss_adversarial.detach().item(),
        "cycle": loss_cycle.detach().item(),
        "identity": loss_identity.detach().item(),
        "cam": loss_cam.detach().item()
    }

In [ ]:
# Run once: these updates become part of this experiment's training budget.
if attention_experiment["diagnostic_history"]:
    raise RuntimeError("Diagnostic updates are already recorded for this experiment.")

attention_step_times = []
attention_config = attention_experiment["config"]

for step, training_batch in enumerate(
    attention_experiment["train_loader"], start=1
):
    # Synchronization makes timing include completed GPU computations.
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

    started = perf_counter()

    losses = attention_train_step(training_batch, attention_experiment, device)

    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = perf_counter() - started
    attention_step_times.append(elapsed)

    attention_experiment["diagnostic_history"].append({
        "step": step,
        "seconds": elapsed,
        **losses
    })

    # Save after every update so completed diagnostic work is recorded.
    pd.DataFrame(
        attention_experiment["diagnostic_history"]
    ).to_csv(
        attention_experiment["run_dir"]
        / "reports/diagnostic_updates.csv",
        index=False
    )

    if step % 5 == 0:
        print(
            f"Step {step:02d} | "
            f"G: {losses['generator']:.3f} | "
            f"D: {losses['discriminator']:.3f} | "
            f"CAM: {losses['cam']:.3f}",
            flush=True
        )

    if step >= attention_config["diagnostic_steps"]:
        break

assert (
    len(attention_experiment["diagnostic_history"])
    == attention_config["diagnostic_steps"]
), "The loader ended before all diagnostic updates completed."

# Exclude the first five updates to reduce warm-up effects.
attention_median_step = float(np.median(attention_step_times[5:]))
attention_epoch_minutes = (
    attention_median_step
    * len(attention_experiment["train_loader"])
    / 60
)

print(f"\nMedian step time: {attention_median_step:.3f} seconds")
print(f"Estimated compute per epoch: {attention_epoch_minutes:.1f} minutes")
print(
    f"Estimated compute for {attention_config['epochs']} epochs: "
    f"{attention_epoch_minutes * attention_config['epochs']:.1f} minutes"
)
print("Estimates exclude plotting, checkpointing, and loading overhead.")

In [ ]:
def train_attention_experiment(experiment, device):
    """Train a fresh attention experiment and save progress each epoch.

    Parameters
    ----------
    experiment : dict
        Prepared experiment with completed diagnostic updates.
    device : torch.device
        Training device.

    Returns
    -------
    pandas.DataFrame
        Epoch-average losses and elapsed times.

    Notes
    -----
    Refuses to restart a run containing a training checkpoint.
    Saved states support a separate resume workflow if interrupted.
    """
    config = experiment["config"]
    run_dir = experiment["run_dir"]
    networks = experiment["models"]
    loader = experiment["train_loader"]
    latest_path = run_dir / "checkpoints/pilot_latest.pt"

    if latest_path.exists():
        raise RuntimeError(
            "A checkpoint already exists. Resume this run rather than "
            "restarting this cell."
        )
    if len(experiment["diagnostic_history"]) != config["diagnostic_steps"]:
        raise RuntimeError("Complete the diagnostic updates first.")

    history = []

    def save_checkpoint(epoch):
        """Save current training state atomically."""
        checkpoint = {
            "epoch": epoch,
            "models": {
                name: model.state_dict()
                for name, model in networks.items()
            },
            "optimizer_G": experiment["optimizer_G"].state_dict(),
            "optimizer_D": experiment["optimizer_D"].state_dict(),
            "history": history,
            "diagnostic_history": experiment["diagnostic_history"],
            "config": config,
            "python_rng": random.getstate(),
            "numpy_rng": np.random.get_state(),
            "torch_rng": torch.get_rng_state(),
            "loader_rng": loader.generator.get_state(),
        }
        if device.type == "mps":
            checkpoint["mps_rng"] = torch.mps.get_rng_state()
        elif device.type == "cuda":
            checkpoint["cuda_rng"] = torch.cuda.get_rng_state_all()

        temporary_path = latest_path.with_suffix(".tmp")
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(latest_path)

    # Preserve the diagnostic-trained state before the longer run begins.
    save_checkpoint(epoch=0)

    for epoch in range(1, config["epochs"] + 1):
        started = perf_counter()
        epoch_losses = []

        for training_batch in loader:
            epoch_losses.append(
                attention_train_step(training_batch, experiment, device)
            )

        averages = pd.DataFrame(epoch_losses).mean().to_dict()

        generator = networks["G_photo_to_anime"]
        generator.eval()
        outputs = []

        with torch.inference_mode():
            for image in experiment["validation_batch"]:
                generated, _, _ = generator(image.unsqueeze(0))
                outputs.append(
                    (generated[0].cpu() * 0.5 + 0.5).clamp(0, 1)
                )

        originals = (
            experiment["validation_batch"].cpu() * 0.5 + 0.5
        ).clamp(0, 1)

        history.append({
            "epoch": epoch,
            "minutes": (perf_counter() - started) / 60,
            **averages,
        })
        experiment["history"] = history

        # Save before plotting so a display error cannot lose this epoch.
        pd.DataFrame(history).to_csv(
            run_dir / "reports/pilot_training_history.csv",
            index=False,
        )
        save_checkpoint(epoch)

        if epoch in (5, 10, 20):
            shutil.copy2(
                latest_path,
                run_dir / f"checkpoints/pilot_epoch_{epoch}.pt",
            )

        fig, axes = plt.subplots(2, len(outputs), figsize=(16, 5))
        for index, output in enumerate(outputs):
            axes[0, index].imshow(originals[index].permute(1, 2, 0))
            axes[1, index].imshow(output.permute(1, 2, 0))
            axes[0, index].axis("off")
            axes[1, index].axis("off")

        fig.suptitle(
            f"Attention model — epoch {epoch} | "
            "top: input; bottom: generated"
        )
        fig.tight_layout()
        fig.savefig(
            run_dir / f"figures/pilot_epoch_{epoch:02d}.png",
            dpi=150,
            bbox_inches="tight",
        )
        plt.show()
        plt.close(fig)

        print(
            f"Epoch {epoch}/{config['epochs']} | "
            f"G: {averages['generator']:.3f} | "
            f"D: {averages['discriminator']:.3f} | "
            f"Cycle: {averages['cycle']:.3f} | "
            f"CAM: {averages['cam']:.3f} | "
            f"Time: {history[-1]['minutes']:.1f} min",
            flush=True
        )

    return pd.DataFrame(history)

In [ ]:
attention_history = train_attention_experiment(
    attention_experiment, device
)

### 14.6. Validation comparison across training checkpoints

The epoch-5, epoch-10, and epoch-20 checkpoints translate the same 16 validation portraits used in the earlier resolution comparison.

Separate evaluation models protect the training networks. Images receive deterministic preprocessing, and inference performs no optimizer updates.

Compare facial coherence, anime styling, expression, skin tone, hair color, accessories, and artifacts across checkpoints. These validation examples inform model development; they do not provide an independent test assessment.

In [ ]:
attention_comparison_outputs = {}
attention_comparison_dir = attention_experiment["run_dir"]

assert len(comparison_rows) == 16

for epoch in (5, 10, 20):
    checkpoint_path = (
        attention_comparison_dir
        / f"checkpoints/pilot_epoch_{epoch}.pt"
    )
    assert checkpoint_path.is_file(), f"Missing: {checkpoint_path}"

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False  # Trusted checkpoint created by this notebook.
    )
    config = checkpoint["config"]
    assert checkpoint["epoch"] == epoch

    # Preserve the CPU RNG state while constructing the evaluation model.
    with torch.random.fork_rng(devices=[]):
        evaluation_generator = AttentionGenerator(
            base_channels=config["base_channels"],
            n_blocks=config["residual_blocks_per_stage"]
        )

    evaluation_generator.load_state_dict(
        checkpoint["models"]["G_photo_to_anime"]
    )
    evaluation_generator.to(device).eval()

    evaluation_transform = transforms.Compose([
        transforms.Resize(
            (config["image_size"], config["image_size"])
        ),
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3)
    ])

    generated_images = []

    with torch.inference_mode():
        for path in comparison_rows["path"]:
            with Image.open(PROJECT_ROOT / path) as image:
                tensor = evaluation_transform(
                    image.convert("RGB")
                ).unsqueeze(0).to(device)

            generated, _, _ = evaluation_generator(tensor)
            assert torch.isfinite(generated).all().item()

            generated_images.append(
                (generated[0].cpu() * 0.5 + 0.5).clamp(0, 1)
            )

    attention_comparison_outputs[epoch] = generated_images
    del evaluation_generator, checkpoint, tensor, generated

comparison_rows.to_csv(
    attention_comparison_dir
    / "reports/validation_comparison_samples.csv",
    index=False
)

# Four portraits per page keep the four comparison rows readable.
for page in range(4):
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))

    for column in range(4):
        sample_index = page * 4 + column
        path = comparison_rows.iloc[sample_index]["path"]

        with Image.open(PROJECT_ROOT / path) as image:
            axes[0, column].imshow(
                image.convert("RGB"), interpolation="nearest"
            )

        axes[0, column].set_title(f"Sample {sample_index + 1}")

        for row, epoch in enumerate((5, 10, 20), start=1):
            output = attention_comparison_outputs[epoch][sample_index]
            axes[row, column].imshow(
                output.permute(1, 2, 0), interpolation="nearest"
            )

    for row, label in enumerate(
        ("Original", "Epoch 5", "Epoch 10", "Epoch 20")
    ):
        for column in range(4):
            axis = axes[row, column]
            axis.set_xticks([])
            axis.set_yticks([])
            for spine in axis.spines.values():
                spine.set_visible(False)
        axes[row, 0].set_ylabel(label, fontsize=12)

    fig.suptitle("Attention-model validation checkpoint comparison")
    fig.tight_layout()
    fig.savefig(
        attention_comparison_dir
        / f"figures/validation_checkpoints_{page + 1}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.savefig(f"attention-model-sampling-{page}.png")
    plt.show()
    plt.close(fig)

### 14.7. Comparing the three model configurations

Compare the compact CycleGAN at 128 × 128 and 256 × 256 with the attention model at 256 × 256. All outputs come from epoch-20 checkpoints and use the same 16 validation portraits.

Equal epoch counts provide equal passes through the training data, but not equal compute budgets. The attention model also changes architecture, losses, and learning rate, so differences cannot be attributed to attention alone.

Assess facial coherence, anime styling, expression, color preservation, and artifacts. The test set remains untouched.

In [ ]:
# These outputs were generated by the earlier comparison cells.
model_comparison_outputs = {
    "CycleGAN 128": resolution_outputs[128],
    "CycleGAN 256": resolution_outputs[256],
    "Attention 256": attention_comparison_outputs[20],
}

assert len(comparison_rows) == 16
for name, outputs in model_comparison_outputs.items():
    assert len(outputs) == 16, f"Unexpected output count for {name}"

for page in range(4):
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))

    for column in range(4):
        sample_index = page * 4 + column
        path = comparison_rows.iloc[sample_index]["path"]

        with Image.open(PROJECT_ROOT / path) as image:
            axes[0, column].imshow(
                image.convert("RGB"), interpolation="nearest"
            )
        axes[0, column].set_title(f"Sample {sample_index + 1}")

        for row, outputs in enumerate(
            model_comparison_outputs.values(), start=1
        ):
            axes[row, column].imshow(
                outputs[sample_index].permute(1, 2, 0),
                interpolation="nearest"
            )

    labels = ["Original", *model_comparison_outputs.keys()]
    for row, label in enumerate(labels):
        for axis in axes[row]:
            axis.set_xticks([])
            axis.set_yticks([])
            for spine in axis.spines.values():
                spine.set_visible(False)
        axes[row, 0].set_ylabel(label, fontsize=11)

    fig.suptitle("Model comparison — epoch 20")
    fig.tight_layout()
    fig.savefig(
        attention_experiment["run_dir"]
        / f"figures/model_comparison_{page + 1}.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.savefig(f"model_comparison_page_{page}.png")
    plt.show()
    plt.close(fig)

### 14.8. Attention-model training diagnostics

Plot the generator's unweighted loss components separately from its total loss. The weighted contributions show how each term contributes numerically to the generator objective; they do not measure its contribution to gradient magnitude or visual quality.

These are training losses, not validation metrics. Their trends must be interpreted alongside generated portraits.

In [ ]:
attention_history_df = pd.read_csv(
    attention_experiment["run_dir"]
    / "reports/pilot_training_history.csv"
)

assert np.isfinite(
    attention_history_df.select_dtypes(include="number").to_numpy()
).all(), "Training history contains non-finite values."

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for axis, columns, title in [
    (axes[0, 0], ["generator"], "Generator total"),
    (axes[0, 1], ["discriminator"], "Discriminator total"),
    (axes[0, 2], ["adversarial"], "Unweighted adversarial"),
    (axes[1, 0], ["cycle", "identity"], "Unweighted reconstruction"),
    (axes[1, 1], ["cam"], "Unweighted attention classification"),
]:
    attention_history_df.plot(
        x="epoch", y=columns, ax=axis, marker="."
    )
    axis.set_title(title)

weighted_losses = pd.DataFrame({
    "epoch": attention_history_df["epoch"],
    "adversarial": (
        attention_history_df["adversarial"]
        * attention_config["lambda_adversarial"]
    ),
    "cycle": (
        attention_history_df["cycle"]
        * attention_config["lambda_cycle"]
    ),
    "identity": (
        attention_history_df["identity"]
        * attention_config["lambda_identity"]
    ),
    "attention classification": (
        attention_history_df["cam"]
        * attention_config["lambda_cam"]
    ),
})
weighted_losses.plot(x="epoch", ax=axes[1, 2])
axes[1, 2].set_title("Weighted generator loss contributions")

for axis in axes.flat:
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Loss")
    axis.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(
    attention_experiment["run_dir"]
    / "figures/training_diagnostics.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

### 14.9. Lower-learning-rate continuation

Continue the attention model from epoch 20 through epoch 30, reducing both optimizer learning rates from 0.0001 to 0.00005. Architecture, loss weights, batch size, and data partitions remain unchanged.

Model weights, optimizer states, and saved random-number states are restored from the epoch-20 checkpoint. No additional diagnostic updates are performed.

This is a new training phase chosen after inspecting validation results, rather than part of the original pilot schedule. The original run is preserved. Checkpoints at epochs 25 and 30 will support comparison with epoch 20.

In [ ]:
def continue_attention_training(experiment, device, end_epoch=30):
    """Continue the epoch-20 attention checkpoint at a lower learning rate.

    Parameters
    ----------
    experiment : dict
        Original attention experiment, including networks and data loader.
    device : torch.device
        Training device.
    end_epoch : int, default=30
        Final epoch number; must exceed 20.

    Returns
    -------
    dict
        Continuation directory, configuration, and combined training history.
    """
    if end_epoch <= 20:
        raise ValueError("end_epoch must exceed 20.")

    parent_dir = Path(experiment["run_dir"])
    source_path = parent_dir / "checkpoints/pilot_epoch_20.pt"

    checkpoint = torch.load(
        source_path, map_location="cpu", weights_only=False
    )
    assert checkpoint["epoch"] == 20

    networks = experiment["models"]
    loader = experiment["train_loader"]

    # Restore all six networks, including discriminator spectral buffers.
    for name, network in networks.items():
        network.load_state_dict(checkpoint["models"][name])
        network.to(device)

    experiment["optimizer_G"].load_state_dict(checkpoint["optimizer_G"])
    experiment["optimizer_D"].load_state_dict(checkpoint["optimizer_D"])

    # Retain Adam's learned moments while reducing its step size.
    new_learning_rate = 0.00005
    for optimizer in (
        experiment["optimizer_G"], experiment["optimizer_D"]
    ):
        for group in optimizer.param_groups:
            group["lr"] = new_learning_rate

    config = dict(checkpoint["config"])
    config.update({
        "epochs": end_epoch,
        "learning_rate": new_learning_rate,
        "lr_schedule": "0.0001 through epoch 20; 0.00005 thereafter",
        "continuation_start_epoch": 21,
        "parent_run": parent_dir.name,
        "parent_checkpoint": "pilot_epoch_20.pt"
    })

    run_id = datetime.now(timezone.utc).strftime("attention_continuation_%Y%m%dT%H%M%S_%fZ")
    run_dir = PROJECT_ROOT / "runs" / run_id
    
    for folder in ("figures", "reports", "checkpoints"):
        (run_dir / folder).mkdir(parents=True, exist_ok=False)

    with (run_dir / "reports/config.json").open("w") as handle:
        json.dump(config, handle, indent=2)

    # Preserve the manifests actually saved with the original experiment.
    for filename in (
        "split_manifest.csv", "fixed_validation_photos.csv",
        "validation_comparison_samples.csv", "diagnostic_updates.csv"
    ):
        source = parent_dir / "reports" / filename
        if source.is_file():
            shutil.copy2(source, run_dir / "reports" / filename)

    continuation = dict(experiment)
    continuation.update({
        "config": config,
        "run_dir": run_dir,
        "diagnostic_history": checkpoint["diagnostic_history"]
    })
    history = [dict(row) for row in checkpoint["history"]]

    # Restore RNG states after preparation, immediately before training.
    random.setstate(checkpoint["python_rng"])
    np.random.set_state(checkpoint["numpy_rng"])
    torch.set_rng_state(checkpoint["torch_rng"])
    loader.generator.set_state(checkpoint["loader_rng"])

    if device.type == "mps" and "mps_rng" in checkpoint:
        torch.mps.set_rng_state(checkpoint["mps_rng"])
    elif device.type == "cuda" and "cuda_rng" in checkpoint:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng"])

    print("Continuation directory:", run_dir, flush=True)

    for epoch in range(21, end_epoch + 1):
        started = perf_counter()
        epoch_losses = []

        for batch in loader:
            epoch_losses.append(
                attention_train_step(batch, continuation, device)
            )

        averages = pd.DataFrame(epoch_losses).mean().to_dict()
        history.append({
            "epoch": epoch,
            "minutes": (perf_counter() - started) / 60,
            **averages,
        })
        pd.DataFrame(history).to_csv(run_dir / "reports/pilot_training_history.csv", index=False)

        state = {
            "epoch": epoch,
            "models": {
                name: network.state_dict()
                for name, network in networks.items()
            },
            "optimizer_G": continuation["optimizer_G"].state_dict(),
            "optimizer_D": continuation["optimizer_D"].state_dict(),
            "history": history,
            "diagnostic_history": continuation["diagnostic_history"],
            "config": config,
            "python_rng": random.getstate(),
            "numpy_rng": np.random.get_state(),
            "torch_rng": torch.get_rng_state(),
            "loader_rng": loader.generator.get_state()
        }
        if device.type == "mps":
            state["mps_rng"] = torch.mps.get_rng_state()
        elif device.type == "cuda":
            state["cuda_rng"] = torch.cuda.get_rng_state_all()

        latest = run_dir / "checkpoints/pilot_latest.pt"
        temporary = latest.with_suffix(".tmp")
        torch.save(state, temporary)
        temporary.replace(latest)

        if epoch in (25, 30):
            shutil.copy2(
                latest, run_dir / f"checkpoints/pilot_epoch_{epoch}.pt"
            )

        # Monitor the same eight portraits without building gradient graphs.
        generator = networks["G_photo_to_anime"]
        generator.eval()
        outputs = []
        with torch.inference_mode():
            for image in continuation["validation_batch"]:
                generated, _, _ = generator(image.unsqueeze(0))
                outputs.append(
                    (generated[0].cpu() * 0.5 + 0.5).clamp(0, 1)
                )

        originals = (
            continuation["validation_batch"].cpu() * 0.5 + 0.5
        ).clamp(0, 1)

        fig, axes = plt.subplots(2, len(outputs), figsize=(16, 5))
        for index, output in enumerate(outputs):
            axes[0, index].imshow(originals[index].permute(1, 2, 0))
            axes[1, index].imshow(output.permute(1, 2, 0))
            axes[0, index].axis("off")
            axes[1, index].axis("off")

        fig.suptitle(f"Attention continuation — epoch {epoch}")
        fig.tight_layout()
        fig.savefig(
            run_dir / f"figures/pilot_epoch_{epoch:02d}.png",
            dpi=150, bbox_inches="tight",
        )
        plt.show()
        plt.close(fig)

        print(
            f"Epoch {epoch}/{end_epoch} | "
            f"G: {averages['generator']:.3f} | "
            f"D: {averages['discriminator']:.3f} | "
            f"Cycle: {averages['cycle']:.3f} | "
            f"CAM: {averages['cam']:.3f} | "
            f"Time: {history[-1]['minutes']:.1f} min",
            flush=True,
        )

    return {
        "run_dir": run_dir,
        "config": config,
        "history": pd.DataFrame(history),
    }

In [ ]:
attention_continuation = continue_attention_training(
    attention_experiment, device, end_epoch=30
)

In [ ]:
continuation_outputs = {}

checkpoint_locations = {
    20: attention_experiment["run_dir"],
    25: attention_continuation["run_dir"],
    30: attention_continuation["run_dir"]
}

for epoch, directory in checkpoint_locations.items():
    checkpoint = torch.load(
        Path(directory) / f"checkpoints/pilot_epoch_{epoch}.pt",
        map_location="cpu",
        weights_only=False  # Trusted checkpoint from this notebook.
    )
    config = checkpoint["config"]
    assert checkpoint["epoch"] == epoch

    with torch.random.fork_rng(devices=[]):
        evaluation_generator = AttentionGenerator(
            base_channels=config["base_channels"],
            n_blocks=config["residual_blocks_per_stage"]
        )

    evaluation_generator.load_state_dict(
        checkpoint["models"]["G_photo_to_anime"]
    )
    evaluation_generator.to(device).eval()

    evaluation_transform = transforms.Compose([
        transforms.Resize(
            (config["image_size"], config["image_size"])
        ),
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3)
    ])

    outputs = []
    with torch.inference_mode():
        for path in comparison_rows["path"]:
            with Image.open(PROJECT_ROOT / path) as image:
                tensor = evaluation_transform(
                    image.convert("RGB")
                ).unsqueeze(0).to(device)

            generated, _, _ = evaluation_generator(tensor)
            assert torch.isfinite(generated).all().item()
            outputs.append(
                (generated[0].cpu() * 0.5 + 0.5).clamp(0, 1)
            )

    continuation_outputs[epoch] = outputs
    del evaluation_generator, checkpoint, tensor, generated

for page in range(4):
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))

    for column in range(4):
        sample_index = page * 4 + column
        path = comparison_rows.iloc[sample_index]["path"]

        with Image.open(PROJECT_ROOT / path) as image:
            axes[0, column].imshow(
                image.convert("RGB"), interpolation="nearest"
            )
        axes[0, column].set_title(f"Sample {sample_index + 1}")

        for row, epoch in enumerate((20, 25, 30), start=1):
            axes[row, column].imshow(
                continuation_outputs[epoch][sample_index].permute(1, 2, 0),
                interpolation="nearest"
            )

    for row, label in enumerate(
        ("Original", "Epoch 20", "Epoch 25", "Epoch 30")
    ):
        for axis in axes[row]:
            axis.set_xticks([])
            axis.set_yticks([])
            for spine in axis.spines.values():
                spine.set_visible(False)
        axes[row, 0].set_ylabel(label, fontsize=12)

    fig.suptitle("Attention model — lower-learning-rate continuation")
    fig.tight_layout()
    fig.savefig(
        attention_continuation["run_dir"]
        / f"figures/continuation_comparison_{page + 1}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

display(
    attention_continuation["history"].loc[
        lambda frame: frame["epoch"].isin([20, 25, 30])
    ]
)

## 15. Revised discriminator experiment

The implementation audit identified several differences between the reduced attention model and the authors’ U-GAT-IT discriminators.

This experiment changes the discriminator configuration while retaining the existing 32-channel generator, data partitions, preprocessing, loss definitions, and initial learning rate.

### Planned changes

- Increase local discriminator depth from 4 to 5 layers.
- Increase global discriminator depth from 6 to 7 layers.
- Use the underlying trainable classifier weights for discriminator channel attention, matching the authors’ implementation.
- Remove spectral normalization from the discriminator's attention-fusion convolution.
- Retain the 512-channel cap as an explicit local-compute adaptation.

The convolutional patch paths will have theoretical receptive fields of 70 × 70 pixels locally and 286 × 286 pixels globally. Global pooling in the attention branches also supplies image-wide information.

### Experimental scope

This is a comparison of discriminator configurations. Because several discriminator properties change together, any improvement cannot be attributed to receptive-field size alone.

Both generators and all discriminators will be initialized from scratch. Starting from the previous generator checkpoint would introduce a different training history and make the comparison harder to interpret.

A short diagnostic will check output shapes, gradients, memory use where measurable, and speed before a training budget is selected. Evaluation will reuse the fixed validation portraits; the test set remains untouched.

### Source

[Authors’ U-GAT-IT network implementation](https://github.com/znxlwm/UGATIT-pytorch/blob/master/networks.py)

In [ ]:
class RevisedAttentionDiscriminator(AttentionDiscriminator):
    """Use deeper patch paths and the authors' attention-weighting convention.

    Parameters
    ----------
    base_channels : int, default=32
        Number of channels in the first convolution.
    n_layers : int, default=5
        Convolutional depth. Use 5 locally and 7 globally.
    max_channels : int, default=512
        Maximum feature width; retained as a compute adaptation.
    """

    def __init__(self, base_channels=32, n_layers=5, max_channels=512):
        super().__init__(
            base_channels=base_channels,
            n_layers=n_layers,
            max_channels=max_channels,
        )

        channels = self.gap_classifier.in_features

        # The authors use an ordinary convolution for attention fusion.
        self.attention_fusion = nn.Conv2d(
            channels * 2, channels, kernel_size=1
        )

    def forward(self, x):
        """Compute patch scores and attention diagnostics.

        Parameters
        ----------
        x : torch.Tensor
            Normalized RGB images shaped (batch, 3, height, width).

        Returns
        -------
        tuple[torch.Tensor, torch.Tensor, torch.Tensor]
            Patch scores, attention logits shaped (batch, 2),
            and a spatial attention heatmap.
        """
        features = self.features(x)

        average = F.adaptive_avg_pool2d(features, 1).flatten(1)
        maximum = F.adaptive_max_pool2d(features, 1).flatten(1)

        # Classifier logits still use spectrally normalized weights.
        average_logit = self.gap_classifier(average)
        maximum_logit = self.gmp_classifier(maximum)
        attention_logits = torch.cat(
            [average_logit, maximum_logit], dim=1
        )

        # Match the authors' use of the underlying trainable weights
        # for channel reweighting, rather than the normalized weights.
        average_features = features * self.gap_classifier.weight_orig[
            :, :, None, None
        ]
        maximum_features = features * self.gmp_classifier.weight_orig[
            :, :, None, None
        ]

        attended = F.leaky_relu(
            self.attention_fusion(
                torch.cat(
                    [average_features, maximum_features], dim=1
                )
            ),
            negative_slope=0.2,
        )

        heatmap = attended.sum(dim=1, keepdim=True)
        return self.patch_head(attended), attention_logits, heatmap

In [ ]:
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(SEED)

    for scale, depth, expected_grid in [
        ("local", 5, 30),
        ("global", 7, 6),
    ]:
        probe = RevisedAttentionDiscriminator(
            base_channels=32,
            n_layers=depth,
            max_channels=512,
        )
        probe.apply(initialize_attention_weights)
        probe.to(device).train()

        # Deterministic input avoids consuming the device's random stream.
        probe_input = torch.linspace(
            -1, 1, steps=3 * 256 * 256
        ).reshape(1, 3, 256, 256).to(device)

        scores, logits, heatmap = probe(probe_input)

        assert scores.shape == (1, 1, expected_grid, expected_grid)
        assert logits.shape == (1, 2)

        for tensor in (scores, logits, heatmap):
            assert torch.isfinite(tensor).all().item()

        # Exercise the patch head and both attention classifiers.
        diagnostic_loss = (
            (scores - 1).square().mean()
            + (logits - 1).square().mean()
        )
        diagnostic_loss.backward()

        for name, parameter in probe.named_parameters():
            assert parameter.grad is not None, f"Missing gradient: {name}"
            assert torch.isfinite(parameter.grad).all().item(), (
                f"Non-finite gradient: {name}"
            )

        assert sum(
            parameter.grad.abs().sum().item()
            for parameter in probe.parameters()
        ) > 0

        print(
            f"{scale.capitalize()} | "
            f"parameters={sum(p.numel() for p in probe.parameters()):,} | "
            f"scores={tuple(scores.shape)} | "
            f"heatmap={tuple(heatmap.shape)} | "
            "finite gradients"
        )

        del probe, probe_input, scores, logits, heatmap
        del diagnostic_loss, parameter, tensor

if device.type == "mps":
    torch.mps.empty_cache()

print("Revised discriminator checks passed.")

In [ ]:
# Copy the original pilot settings; continuation used a separate config.
revised_config = dict(attention_experiment["config"])
revised_config.update({
    "architecture": "attention generator with revised discriminators",
    "local_discriminator_layers": 5,
    "global_discriminator_layers": 7,
    "discriminator_attention_weights": "weight_orig",
    "discriminator_fusion_spectral_norm": False,
    "learning_rate": 0.0001,
    "epochs": 20,  # Provisional budget, pending the timing diagnostic.
    "lr_schedule": "constant during initial pilot",
})

random.seed(revised_config["seed"])
np.random.seed(revised_config["seed"])
torch.manual_seed(revised_config["seed"])

revised_dataset = UnpairedPortraitDataset(
    split_manifest.loc[split_manifest["split"].eq("train")],
    PROJECT_ROOT,
    attention_train_transform,
)
revised_loader = DataLoader(
    revised_dataset,
    batch_size=revised_config["batch_size"],
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(revised_config["seed"]),
)

revised_models = {
    "G_photo_to_anime": AttentionGenerator(
        revised_config["base_channels"],
        revised_config["residual_blocks_per_stage"],
    ),
    "G_anime_to_photo": AttentionGenerator(
        revised_config["base_channels"],
        revised_config["residual_blocks_per_stage"],
    ),
}

# Initialize the generators before constructing discriminators so that
# discriminator depth does not affect their initialization random stream.
for name, network in revised_models.items():
    network.apply(initialize_attention_weights)
    network.to(device)

for domain in ("photo", "anime"):
    for scale, depth in (("local", 5), ("global", 7)):
        network = RevisedAttentionDiscriminator(
            base_channels=revised_config["base_channels"],
            n_layers=depth,
            max_channels=revised_config["max_discriminator_channels"],
        )
        network.apply(initialize_attention_weights)
        revised_models[f"D_{domain}_{scale}"] = network.to(device)

revised_optimizer_G = torch.optim.Adam(
    chain(
        revised_models["G_photo_to_anime"].parameters(),
        revised_models["G_anime_to_photo"].parameters(),
    ),
    lr=revised_config["learning_rate"],
    betas=tuple(revised_config["adam_betas"]),
)
revised_optimizer_D = torch.optim.Adam(
    chain.from_iterable(
        network.parameters()
        for name, network in revised_models.items()
        if name.startswith("D_")
    ),
    lr=revised_config["learning_rate"],
    betas=tuple(revised_config["adam_betas"]),
)

revised_run_id = datetime.now(timezone.utc).strftime(
    "revised_discriminators_%Y%m%dT%H%M%S_%fZ"
)
revised_run_dir = PROJECT_ROOT / "runs" / revised_run_id

for folder in ("figures", "reports", "checkpoints"):
    (revised_run_dir / folder).mkdir(parents=True, exist_ok=False)

with (revised_run_dir / "reports/config.json").open("w") as handle:
    json.dump(revised_config, handle, indent=2)

for filename, manifest in [
    ("split_manifest.csv", split_manifest),
    ("fixed_validation_photos.csv", validation_photos),
    ("validation_comparison_samples.csv", comparison_rows),
]:
    manifest.to_csv(
        revised_run_dir / "reports" / filename, index=False
    )

revised_experiment = {
    "config": revised_config,
    "run_dir": revised_run_dir,
    "models": revised_models,
    "optimizer_G": revised_optimizer_G,
    "optimizer_D": revised_optimizer_D,
    "train_loader": revised_loader,
    "validation_batch": attention_validation_batch,
    "diagnostic_history": [],
}

for name, network in revised_models.items():
    print(f"{name}: {sum(p.numel() for p in network.parameters()):,} parameters")

print("\nRun directory:", revised_run_dir)
print("Prepared fresh networks; no training updates performed.")

In [ ]:
if revised_experiment["diagnostic_history"]:
    raise RuntimeError("Diagnostic updates are already recorded.")

revised_step_times = []

for step, batch in enumerate(
    revised_experiment["train_loader"], start=1
):
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

    started = perf_counter()
    losses = attention_train_step(batch, revised_experiment, device)

    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = perf_counter() - started
    revised_step_times.append(elapsed)
    revised_experiment["diagnostic_history"].append({
        "step": step,
        "seconds": elapsed,
        **losses,
    })

    pd.DataFrame(
        revised_experiment["diagnostic_history"]
    ).to_csv(
        revised_experiment["run_dir"] / "reports/diagnostic_updates.csv",
        index=False,
    )

    if step % 5 == 0:
        print(
            f"Step {step:02d} | "
            f"G: {losses['generator']:.3f} | "
            f"D: {losses['discriminator']:.3f} | "
            f"CAM: {losses['cam']:.3f}",
            flush=True,
        )

    if step >= revised_experiment["config"]["diagnostic_steps"]:
        break

assert len(revised_step_times) == revised_experiment["config"]["diagnostic_steps"]

median_seconds = float(np.median(revised_step_times[5:]))
epoch_minutes = (
    median_seconds * len(revised_experiment["train_loader"]) / 60
)

print(f"\nMedian step time: {median_seconds:.3f} seconds")
print(f"Estimated compute per epoch: {epoch_minutes:.1f} minutes")
print(f"Estimated compute for 20 epochs: {epoch_minutes * 20:.1f} minutes")
print("Estimates exclude plotting, checkpointing, and loading overhead.")

if device.type == "mps":
    print(
        "Current MPS tensor memory: "
        f"{torch.mps.current_allocated_memory() / 1024**3:.2f} GiB"
    )

In [ ]:
revised_history = train_attention_experiment(
    revised_experiment, device
)

### 15.1. Validation comparison of discriminator configurations

Both epoch-20 checkpoints translate the same 16 validation portraits. Separate evaluation generators protect the training networks.

Compare facial coherence, expression, color preservation, anime styling, and artifacts. This comparison covers a bundle of discriminator changes and different generator initializations; it does not isolate a single cause. The test set remains untouched.

In [ ]:
discriminator_comparison_outputs = {}

comparison_runs = {
    "Original attention": Path(attention_experiment["run_dir"]),
    "Revised discriminators": Path(revised_experiment["run_dir"]),
}

for label, run_directory in comparison_runs.items():
    checkpoint = torch.load(
        run_directory / "checkpoints/pilot_epoch_20.pt",
        map_location="cpu",
        weights_only=False,  # Trusted checkpoint created by this notebook.
    )
    config = checkpoint["config"]
    assert checkpoint["epoch"] == 20

    with torch.random.fork_rng(devices=[]):
        evaluation_generator = AttentionGenerator(
            base_channels=config["base_channels"],
            n_blocks=config["residual_blocks_per_stage"],
        )

    evaluation_generator.load_state_dict(
        checkpoint["models"]["G_photo_to_anime"]
    )
    evaluation_generator.to(device).eval()

    evaluation_transform = transforms.Compose([
        transforms.Resize(
            (config["image_size"], config["image_size"])
        ),
        transforms.ToTensor(),
        transforms.Normalize((0.5,) * 3, (0.5,) * 3),
    ])

    outputs = []
    with torch.inference_mode():
        for path in comparison_rows["path"]:
            with Image.open(PROJECT_ROOT / path) as image:
                tensor = evaluation_transform(
                    image.convert("RGB")
                ).unsqueeze(0).to(device)

            generated, _, _ = evaluation_generator(tensor)
            assert torch.isfinite(generated).all().item()
            outputs.append(
                (generated[0].cpu() * 0.5 + 0.5).clamp(0, 1)
            )

    discriminator_comparison_outputs[label] = outputs
    del evaluation_generator, checkpoint, tensor, generated

for page in range(4):
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))

    for column in range(4):
        sample_index = page * 4 + column
        path = comparison_rows.iloc[sample_index]["path"]

        with Image.open(PROJECT_ROOT / path) as image:
            axes[0, column].imshow(
                image.convert("RGB"), interpolation="nearest"
            )
        axes[0, column].set_title(f"Sample {sample_index + 1}")

        for row, outputs in enumerate(
            discriminator_comparison_outputs.values(), start=1
        ):
            axes[row, column].imshow(
                outputs[sample_index].permute(1, 2, 0),
                interpolation="nearest",
            )

    labels = ["Original photo", *discriminator_comparison_outputs]
    for row, label in enumerate(labels):
        for axis in axes[row]:
            axis.set_xticks([])
            axis.set_yticks([])
            for spine in axis.spines.values():
                spine.set_visible(False)
        axes[row, 0].set_ylabel(label, fontsize=11)

    fig.suptitle("Discriminator configuration comparison — epoch 20")
    fig.tight_layout()
    fig.savefig(
        revised_experiment["run_dir"]
        / f"figures/discriminator_comparison_{page + 1}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

In [ ]:
revised_experiment["run_dir"]